# EEG Abnormality Prediction

Template-driven healthcare machine learning notebook for the HCTM EEG Abnormality project. This notebook is research-only and is not intended for clinical deployment, diagnosis, or treatment decisions.

# 1. Data Loading

This section defines the EEG project context, imports libraries, loads the HCTM dataset, and constructs the binary EEG abnormality target.

## 1.1 Project Setup

In [ ]:
# Core libraries
import os
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
SHAP_OUTPUT_DIR = OUTPUT_DIR / "shap"
MPLCONFIG_DIR = OUTPUT_DIR / "matplotlib"
for directory in [OUTPUT_DIR, SHAP_OUTPUT_DIR, MPLCONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(MPLCONFIG_DIR)
os.environ["PYTHONHASHSEED"] = "42"
warnings.filterwarnings("ignore")

import numpy as np
if not hasattr(np, "NaN"):
    np.NaN = np.nan
import pandas as pd
import scipy.stats as scipy_stats

if not hasattr(scipy_stats, "binom_test"):
    scipy_stats.binom_test = lambda x, n=None, p=0.5, alternative="two-sided": scipy_stats.binomtest(
        int(round(x)), n=int(n), p=p, alternative=alternative
    ).pvalue

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC

from boruta import BorutaPy
from BorutaShap import BorutaShap
from catboost import CatBoostClassifier, Pool
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")

print(f"Project root: {PROJECT_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Random seed: {RANDOM_STATE}")

## 1.2 Project Configuration

Project-specific constants are defined here. The remaining notebook follows the healthcare ML template workflow.

In [ ]:
PROJECT_NAME = "EEG Abnormality Prediction"
SHORT_NAME = "EEG Abnormality"
DATA_PATH = Path("Dataset/EEG ABNORMALITY DATASET.xlsx")
SHEET_NAME = "Raw Data Set"
TARGET_COLUMN = "EEG FINDINGS"
TARGET_BINARY = "EEG_ABNORMAL"
POSITIVE_CLASS_LABEL = "Abnormal EEG"
TEST_SIZE = 0.20
CV_FOLDS = 5
BORUTASHAP_N_TRIALS = 100
BORUTASHAP_PERCENTILE = 100
BORUTASHAP_PVALUE = 0.05
BORUTASHAP_CATBOOST_ITERATIONS = 80

DATASET_SOURCE = "Hospital Canselor Tuanku Muhriz (HCTM)"
RESEARCH_ONLY_NOTE = (
    "Research-only retrospective single-centre analysis. "
    "No clinical deployment, diagnosis, or treatment decision claims are made."
)

config_summary = pd.DataFrame(
    {
        "item": [
            "project",
            "dataset_source",
            "dataset_path",
            "sheet_name",
            "target_column",
            "binary_target",
            "positive_class",
            "test_size",
            "cv_folds",
        ],
        "value": [
            PROJECT_NAME,
            DATASET_SOURCE,
            str(DATA_PATH),
            SHEET_NAME,
            TARGET_COLUMN,
            TARGET_BINARY,
            POSITIVE_CLASS_LABEL,
            TEST_SIZE,
            CV_FOLDS,
        ],
    }
)
display(config_summary)

## 1.3 Load Dataset

In [ ]:
def load_dataset(path, sheet_name=None):
    suffix = path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path, sheet_name=sheet_name)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported dataset suffix: {suffix}")


if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

raw_data = load_dataset(DATA_PATH, SHEET_NAME)
raw_shape = raw_data.shape
raw_columns_original = list(raw_data.columns)
raw_data = raw_data.copy()
raw_data.columns = raw_data.columns.astype(str).str.strip()
raw_columns_cleaned = list(raw_data.columns)

column_check = pd.DataFrame(
    {
        "column": raw_columns_cleaned,
        "dtype": [str(raw_data[col].dtype) for col in raw_columns_cleaned],
        "missing_n": [int(raw_data[col].isna().sum()) for col in raw_columns_cleaned],
        "missing_pct": [float(raw_data[col].isna().mean() * 100) for col in raw_columns_cleaned],
    }
)

changed_columns = [before for before, after in zip(raw_columns_original, raw_columns_cleaned) if before != after]
print(f"Raw dataset shape: {raw_shape[0]} rows x {raw_shape[1]} columns")
print("Column-name whitespace was stripped in memory only; the raw workbook was not edited.")
print(f"Columns changed after stripping whitespace: {changed_columns}")
display(column_check)

## 1.4 Target Definition

The target maps `Normal` EEG findings to 0 and all clinically non-normal EEG findings to 1. Rows with missing or invalid target values are removed before mapping.

In [ ]:
data = raw_data.copy()

if TARGET_COLUMN not in data.columns:
    raise KeyError(f"Required target column not found: {TARGET_COLUMN}")


def clean_text_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in {"", "-", "NA", "N/A", "None", "NULL", "nan"}:
        return np.nan
    return text


target_clean = data[TARGET_COLUMN].map(clean_text_value)
valid_target_mask = target_clean.notna()
invalid_target_rows = int((~valid_target_mask).sum())
if invalid_target_rows:
    print(f"Dropping {invalid_target_rows} rows with missing/invalid target values.")

data = data.loc[valid_target_mask].copy()
target_clean = target_clean.loc[valid_target_mask]
data[TARGET_BINARY] = np.where(target_clean.str.lower().eq("normal"), 0, 1).astype(int)

raw_target_distribution = target_clean.value_counts(dropna=False).rename_axis("raw_eeg_finding").reset_index(name="n")
binary_target_distribution = (
    data[TARGET_BINARY]
    .map({0: "Normal EEG (0)", 1: "Abnormal EEG (1)"})
    .value_counts()
    .rename_axis("target_class")
    .reset_index(name="n")
)
binary_target_distribution["pct"] = binary_target_distribution["n"] / binary_target_distribution["n"].sum() * 100

print(f"Analysis dataset shape after target cleaning: {data.shape[0]} rows x {data.shape[1]} columns")
display(raw_target_distribution)
display(binary_target_distribution)

# 2. Preprocessing

This section implements leakage-aware feature exclusion, EDA-driven cleaning, stratified splitting, train-fitted preprocessing, and model-specific feature selection before benchmarking.

## 2.1 Feature Exclusion

The following exclusions remove leakage, downstream outcomes, post-hoc interpretations, unavailable fields, high-missing free text, and redundant raw fields. This is feature exclusion, not feature selection.

In [ ]:
LEAKAGE_COLUMNS = [
    "CAUSES",
    "LENGTH OF HOSPITALISATION",
    "OUTCOME OF HOSPITALISATION",
    "INDICATION FOR EEG",
    "DIAGNOSIS",
    "DXT/FBS/RBS/HbA1c",
    "OTHERS",
    "TOTAL CHOLESTEROL",
    "BLOOD PRESSURE ON PRESENTATION",
]

exclusion_reasons = {
    "CAUSES": "Leakage / diagnostic pathway / possible post-hoc clinical interpretation",
    "LENGTH OF HOSPITALISATION": "Post-outcome / downstream variable",
    "OUTCOME OF HOSPITALISATION": "Post-outcome",
    "INDICATION FOR EEG": "EEG-ordering pathway / potential target proxy",
    "DIAGNOSIS": "Target proxy / post-clinical interpretation",
    "DXT/FBS/RBS/HbA1c": "Excluded raw/related glucose measurement field",
    "OTHERS": "99 percent missing / free-text / low utility",
    "TOTAL CHOLESTEROL": "Excluded based on prior redundancy/project decision",
    "BLOOD PRESSURE ON PRESENTATION": "Raw combined BP field; separated BP fields are used if retained",
}

present_leakage_columns = [col for col in LEAKAGE_COLUMNS if col in data.columns]
missing_leakage_columns = [col for col in LEAKAGE_COLUMNS if col not in data.columns]

exclusion_summary = pd.DataFrame(
    {
        "column": LEAKAGE_COLUMNS,
        "present_in_dataset": [col in data.columns for col in LEAKAGE_COLUMNS],
        "reason": [exclusion_reasons[col] for col in LEAKAGE_COLUMNS],
    }
)

display(exclusion_summary)
print(f"Excluded leakage/downstream/unavailable columns present: {len(present_leakage_columns)}")
print(f"Specified exclusion columns not present: {missing_leakage_columns}")


## 2.2 Feature Type Split/Definition

Feature groups are defined explicitly so demographic categories, binary clinical status variables, laboratory values, and CatBoost categorical columns are handled according to the project context.

In [ ]:
numeric_columns_spec = [
    "AGE",
    "Systolic",
    "Diastolic",
    "HAEMOGLOBIN",
    "PLATELET COUNT",
    "WHITE CELL COUNT",
    "BLOOD UREA",
    "POTASSIUM",
    "CREATININE",
    "SODIUM",
    "TOTAL PROTEIN",
    "ALBUMIN",
    "ALANINE TRANSAMINASE",
    "CALCIUM",
    "TRIGLYCERIDE(TG)",
    "LDL",
    "HDL",
]

binary_clinical_categorical_spec = [
    "SMOKING",
    "ALCOHOL",
    "HYPERTENSION",
    "DIABETES MELLITUS",
    "HYPERLIPIDEMIA",
    "STROKE",
    "TB",
    "AF",
    "IHD",
    "CONGESTIVE CARDIAC FAILURE",
    "COPD",
    "BRONCHIAL ASTHMA",
    "CKD",
    "PSYCHIATRIC DISORDER",
    "BRAIN TUMOUR",
    "HIV",
    "SEIZURES",
    "FOCAL SEIZURE",
    "GENERALISED SEIZURE",
]

demographic_categorical_spec = ["GENDER", "RACE"]
nonbinary_clinical_categorical_spec = ["Blood Sugar Control", "RADIOLOGICAL IMAGING"]

numeric_columns = [col for col in numeric_columns_spec if col in data.columns]
binary_clinical_categorical_columns = [col for col in binary_clinical_categorical_spec if col in data.columns]
demographic_categorical_columns = [col for col in demographic_categorical_spec if col in data.columns]
nonbinary_clinical_categorical_columns = [col for col in nonbinary_clinical_categorical_spec if col in data.columns]
categorical_columns = (
    binary_clinical_categorical_columns
    + demographic_categorical_columns
    + nonbinary_clinical_categorical_columns
)

model_data = data.drop(columns=present_leakage_columns, errors="ignore").copy()
feature_columns = [
    col
    for col in numeric_columns + categorical_columns
    if col in model_data.columns and col not in {TARGET_COLUMN, TARGET_BINARY}
]

missing_expected = sorted(
    set(numeric_columns_spec + binary_clinical_categorical_spec + demographic_categorical_spec + nonbinary_clinical_categorical_spec)
    - set(feature_columns)
    - set(LEAKAGE_COLUMNS)
)

feature_type_summary = pd.DataFrame(
    [
        {"feature_group": "numerical", "n": len(numeric_columns), "features": numeric_columns},
        {
            "feature_group": "binary_clinical_categorical",
            "n": len(binary_clinical_categorical_columns),
            "features": binary_clinical_categorical_columns,
        },
        {"feature_group": "demographic_categorical", "n": len(demographic_categorical_columns), "features": demographic_categorical_columns},
        {
            "feature_group": "nonbinary_clinical_categorical",
            "n": len(nonbinary_clinical_categorical_columns),
            "features": nonbinary_clinical_categorical_columns,
        },
    ]
)

print(f"Retained modeling features before feature selection: {len(feature_columns)}")
print(f"Missing expected retained features after in-memory column cleanup: {missing_expected}")
display(feature_type_summary)


## 2.3 EDA-Driven Data Cleaning

The missingness and outlier policies below are implemented using training-set fitted values only. Medical outliers are not deleted blindly.

In [ ]:
numerical_missingness_report = pd.DataFrame(
    [
        ("HAEMOGLOBIN", 3, 1.041667),
        ("PLATELET COUNT", 3, 1.041667),
        ("WHITE CELL COUNT", 3, 1.041667),
        ("BLOOD UREA", 4, 1.388889),
        ("POTASSIUM", 2, 0.694444),
        ("CREATININE", 2, 0.694444),
        ("SODIUM", 2, 0.694444),
        ("TOTAL PROTEIN", 5, 1.736111),
        ("ALBUMIN", 3, 1.041667),
        ("ALANINE TRANSAMINASE", 4, 1.388889),
        ("CALCIUM", 4, 1.388889),
        ("TRIGLYCERIDE(TG)", 52, 18.055556),
        ("LDL", 51, 17.708333),
        ("HDL", 52, 18.055556),
    ],
    columns=["feature", "missing_values", "missing_pct"],
)

categorical_missingness_report = pd.DataFrame(
    [
        ("RADIOLOGICAL IMAGING", 13, 4.513889, "Candidate predictor; preserve imaging findings / Not performed category"),
        ("Blood Sugar Control", 42, 14.583333, "Missing/not tested/not documented as 2"),
        ("DXT/FBS/RBS/HbA1c", 40, 13.888889, "Excluded"),
        ("OTHERS", 286, 99.305556, "Excluded"),
    ],
    columns=["feature", "missing_values", "missing_pct", "handling"],
)

outlier_policy = pd.DataFrame(
    [
        ("AGE", "Retain clinically plausible values"),
        ("Systolic", "Retain clinically plausible values"),
        ("Diastolic", "Retain clinically plausible values"),
        ("PLATELET COUNT", "Train-derived tail capping and log transform if skewed"),
        ("WHITE CELL COUNT", "Train-derived tail capping and log transform if skewed"),
        ("BLOOD UREA", "Train-derived tail capping and log transform if skewed"),
        ("CREATININE", "Train-derived tail capping and log transform if skewed"),
        ("ALANINE TRANSAMINASE", "Train-derived tail capping and log transform if skewed"),
        ("TRIGLYCERIDE(TG)", "Missingness flag, median imputation, log transform if skewed"),
        ("LDL", "Missingness flag and retain/cap only if invalid"),
        ("HDL", "Missingness flag and retain/cap only if invalid"),
    ],
    columns=["feature", "handling"],
)

missingness_policy = pd.DataFrame(
    [
        ("Low-missing numerical features", "Training-set median"),
        ("TRIGLYCERIDE(TG), LDL, HDL", "Training-set median plus missingness flag"),
        ("Binary clinical categorical features", "Missing/unmapped as 2 where clinically appropriate"),
        ("Demographic categorical features", "Training-set mode imputation if missing"),
        ("Blood Sugar Control", "Missing/unknown/not tested as 2"),
        ("RADIOLOGICAL IMAGING", "Candidate predictor; preserve imaging findings and missing/not performed category"),
    ],
    columns=["feature_or_group", "handling_rule"],
)

display(numerical_missingness_report)
display(categorical_missingness_report)
display(outlier_policy)
display(missingness_policy)


In [ ]:
MISSING_TOKENS = {"", " ", "-", "NA", "N/A", "na", "n/a", "None", "none", "NULL", "null", "nan", "NaN"}
YES_TOKENS = {"yes", "y", "true", "1", "present", "positive"}
NO_TOKENS = {"no", "n", "false", "0", "absent", "negative"}
UNKNOWN_TOKENS = {"unknown", "not known", "not documented", "not done", "not performed", "not tested", "not applicable", "missing"}

LIPID_MISSING_FLAG_FEATURES = ["TRIGLYCERIDE(TG)", "LDL", "HDL"]
CAPPING_CANDIDATES = ["PLATELET COUNT", "WHITE CELL COUNT", "BLOOD UREA", "CREATININE", "ALANINE TRANSAMINASE"]
LOG_TRANSFORM_CANDIDATES = CAPPING_CANDIDATES + ["TRIGLYCERIDE(TG)"]


def normalize_missing_tokens(df):
    df = df.copy()
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].map(lambda value: np.nan if pd.isna(value) or str(value).strip() in MISSING_TOKENS else str(value).strip())
    return df


def coerce_numeric(series):
    cleaned = series.map(lambda value: np.nan if pd.isna(value) or str(value).strip() in MISSING_TOKENS else value)
    cleaned = cleaned.astype("string").str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce").astype(float)


def map_binary_clinical_value(value):
    if pd.isna(value):
        return "2"
    text = str(value).strip().lower()
    if text in MISSING_TOKENS or text in UNKNOWN_TOKENS:
        return "2"
    if text in YES_TOKENS:
        return "1"
    if text in NO_TOKENS:
        return "0"
    return "2"


def clean_category_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    if text in MISSING_TOKENS or text.lower() in UNKNOWN_TOKENS:
        return np.nan
    return text


class ClinicalPreprocessor:
    def __init__(
        self,
        numeric_columns,
        binary_categorical_columns,
        demographic_categorical_columns,
        nonbinary_categorical_columns,
        missing_flag_features,
        capping_candidates,
        log_transform_candidates,
    ):
        self.numeric_columns = list(numeric_columns)
        self.binary_categorical_columns = list(binary_categorical_columns)
        self.demographic_categorical_columns = list(demographic_categorical_columns)
        self.nonbinary_categorical_columns = list(nonbinary_categorical_columns)
        self.missing_flag_features = [col for col in missing_flag_features if col in self.numeric_columns]
        self.missing_flag_columns = [f"{col}__missing_flag" for col in self.missing_flag_features]
        self.capping_candidates = [col for col in capping_candidates if col in self.numeric_columns]
        self.log_transform_candidates = [col for col in log_transform_candidates if col in self.numeric_columns]
        self.numeric_medians_ = {}
        self.categorical_modes_ = {}
        self.cap_bounds_ = {}
        self.log_transform_columns_ = []

    def fit(self, X, y=None):
        X_norm = normalize_missing_tokens(X)
        numeric_train = pd.DataFrame(index=X_norm.index)
        for col in self.numeric_columns:
            numeric_train[col] = coerce_numeric(X_norm[col])
            median = numeric_train[col].median()
            self.numeric_medians_[col] = 0.0 if pd.isna(median) else float(median)
            if col in self.capping_candidates:
                observed = numeric_train[col].dropna()
                lower = self.numeric_medians_[col] if observed.empty else float(observed.quantile(0.01))
                upper = self.numeric_medians_[col] if observed.empty else float(observed.quantile(0.99))
                if lower > upper:
                    lower, upper = upper, lower
                self.cap_bounds_[col] = (lower, upper)

        for col in self.demographic_categorical_columns:
            cleaned = X_norm[col].map(clean_category_value)
            mode_values = cleaned.dropna().mode()
            self.categorical_modes_[col] = "Unknown" if mode_values.empty else str(mode_values.iloc[0])

        numeric_for_skew = numeric_train.copy()
        for col in self.numeric_columns:
            numeric_for_skew[col] = numeric_for_skew[col].fillna(self.numeric_medians_[col])
            if col in self.cap_bounds_:
                lower, upper = self.cap_bounds_[col]
                numeric_for_skew[col] = numeric_for_skew[col].clip(lower=lower, upper=upper)

        self.log_transform_columns_ = [
            col
            for col in self.log_transform_candidates
            if numeric_for_skew[col].min() >= 0 and abs(numeric_for_skew[col].skew()) >= 1.0
        ]
        self.numeric_model_columns_ = self.numeric_columns + self.missing_flag_columns
        self.categorical_model_columns_ = (
            self.binary_categorical_columns
            + self.demographic_categorical_columns
            + self.nonbinary_categorical_columns
        )
        self.feature_order_ = self.numeric_model_columns_ + self.categorical_model_columns_
        return self

    def transform(self, X):
        X_norm = normalize_missing_tokens(X)
        transformed = pd.DataFrame(index=X_norm.index)
        for col in self.numeric_columns:
            numeric = coerce_numeric(X_norm[col])
            if col in self.missing_flag_features:
                transformed[f"{col}__missing_flag"] = numeric.isna().astype(int)
            numeric = numeric.fillna(self.numeric_medians_[col])
            if col in self.cap_bounds_:
                lower, upper = self.cap_bounds_[col]
                numeric = numeric.clip(lower=lower, upper=upper)
            if col in self.log_transform_columns_:
                numeric = np.log1p(numeric.clip(lower=0))
            transformed[col] = numeric.astype(float)
        for col in self.binary_categorical_columns:
            transformed[col] = X_norm[col].map(map_binary_clinical_value).astype(str)
        for col in self.demographic_categorical_columns:
            transformed[col] = X_norm[col].map(clean_category_value).fillna(self.categorical_modes_[col]).astype(str)
        for col in self.nonbinary_categorical_columns:
            transformed[col] = X_norm[col].map(clean_category_value).fillna("2").astype(str)
        return transformed[self.feature_order_].copy()

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


print("Cleaning helpers prepared.")

## 2.4 Stratified Train-Test Split

The split is performed before fitting imputation, capping, transformations, encoders, scalers, feature selectors, tuning, or threshold selection.

In [ ]:
X = model_data[feature_columns].copy()
y = model_data[TARGET_BINARY].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    [
        {"split": "train", "n": len(y_train), "abnormal_n": int(y_train.sum()), "abnormal_pct": float(y_train.mean() * 100)},
        {"split": "held_out_test", "n": len(y_test), "abnormal_n": int(y_test.sum()), "abnormal_pct": float(y_test.mean() * 100)},
    ]
)
print("Stratified split complete. Held-out test data is reserved for final evaluation only.")
display(split_summary)

## 2.5 Preprocessing Pipeline Execution

A shared clinical cleaner is fit on the training set, then model-specific matrices are created for linear, tree, and CatBoost models.

In [ ]:
def make_ohe_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


class ModelMatrixBuilder:
    def __init__(self, numeric_columns, categorical_columns):
        self.numeric_columns = list(numeric_columns)
        self.categorical_columns = list(categorical_columns)
        self.scaler_ = StandardScaler()
        self.ohe_ = make_ohe_encoder()

    def fit(self, X_clean):
        self.scaler_.fit(X_clean[self.numeric_columns])
        self.ohe_.fit(X_clean[self.categorical_columns])
        self.encoded_categorical_columns_ = list(self.ohe_.get_feature_names_out(self.categorical_columns))
        self.linear_feature_names_ = self.numeric_columns + self.encoded_categorical_columns_
        self.tree_feature_names_ = self.numeric_columns + self.encoded_categorical_columns_
        return self

    def transform_linear(self, X_clean):
        numeric_scaled = pd.DataFrame(
            self.scaler_.transform(X_clean[self.numeric_columns]),
            columns=self.numeric_columns,
            index=X_clean.index,
        )
        categorical_encoded = pd.DataFrame(
            self.ohe_.transform(X_clean[self.categorical_columns]),
            columns=self.encoded_categorical_columns_,
            index=X_clean.index,
        )
        return pd.concat([numeric_scaled, categorical_encoded], axis=1)[self.linear_feature_names_]

    def transform_tree(self, X_clean):
        numeric_unscaled = X_clean[self.numeric_columns].astype(float).copy()
        categorical_encoded = pd.DataFrame(
            self.ohe_.transform(X_clean[self.categorical_columns]),
            columns=self.encoded_categorical_columns_,
            index=X_clean.index,
        )
        return pd.concat([numeric_unscaled, categorical_encoded], axis=1)[self.tree_feature_names_]


def prepare_catboost_frame(X_clean, categorical_columns):
    X_cb = X_clean.copy()
    for col in categorical_columns:
        X_cb[col] = X_cb[col].astype(str)
    return X_cb


print("Model matrix builder prepared.")

In [ ]:
clinical_preprocessor = ClinicalPreprocessor(
    numeric_columns=numeric_columns,
    binary_categorical_columns=binary_clinical_categorical_columns,
    demographic_categorical_columns=demographic_categorical_columns,
    nonbinary_categorical_columns=nonbinary_clinical_categorical_columns,
    missing_flag_features=LIPID_MISSING_FLAG_FEATURES,
    capping_candidates=CAPPING_CANDIDATES,
    log_transform_candidates=LOG_TRANSFORM_CANDIDATES,
)

X_train_clean = clinical_preprocessor.fit_transform(X_train_raw, y_train)
X_test_clean = clinical_preprocessor.transform(X_test_raw)

numeric_model_columns = clinical_preprocessor.numeric_model_columns_
categorical_model_columns = clinical_preprocessor.categorical_model_columns_
catboost_categorical_features = list(categorical_model_columns)

matrix_builder = ModelMatrixBuilder(numeric_model_columns, categorical_model_columns).fit(X_train_clean)
X_train_linear = matrix_builder.transform_linear(X_train_clean)
X_test_linear = matrix_builder.transform_linear(X_test_clean)
X_train_tree = matrix_builder.transform_tree(X_train_clean)
X_test_tree = matrix_builder.transform_tree(X_test_clean)
X_train_catboost = prepare_catboost_frame(X_train_clean, catboost_categorical_features)
X_test_catboost = prepare_catboost_frame(X_test_clean, catboost_categorical_features)

preprocessing_summary = pd.DataFrame(
    [
        {"artifact": "clean_train_shape", "value": X_train_clean.shape},
        {"artifact": "linear_train_shape", "value": X_train_linear.shape},
        {"artifact": "tree_train_shape", "value": X_train_tree.shape},
        {"artifact": "catboost_train_shape", "value": X_train_catboost.shape},
        {"artifact": "numeric_model_columns", "value": len(numeric_model_columns)},
        {"artifact": "categorical_model_columns", "value": len(categorical_model_columns)},
        {"artifact": "log_transformed_columns", "value": clinical_preprocessor.log_transform_columns_},
        {"artifact": "train_derived_cap_bounds", "value": clinical_preprocessor.cap_bounds_},
    ]
)
display(preprocessing_summary)

## 2.6 Feature Selection

Feature selection is performed before benchmarking and uses training data only. Feature exclusion above is not counted as feature selection.

CatBoost uses BorutaSHAP for all-relevant ML risk-factor discovery. Tentative BorutaSHAP features are reported separately and are not included in the main CatBoost model unless explicitly promoted by a predefined rule.

In [ ]:
FEATURE_SELECTION_METHODS = {
    "logistic_regression": "L1/Lasso LogisticRegressionCV on scaled one-hot training matrix",
    "linear_svm": "RFECV with LinearSVC on scaled one-hot training matrix",
    "random_forest": "Boruta with RandomForestClassifier on unscaled one-hot training matrix",
    "catboost": "BorutaSHAP all-relevant feature selection with CatBoostClassifier on train-only CatBoost matrix",
}

selected_features = {}
selected_feature_counts = {}
feature_selection_details = []
selected_catboost_categorical_features = []
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

display(pd.DataFrame([{"model": key, "method": value} for key, value in FEATURE_SELECTION_METHODS.items()]))

In [ ]:
l1_selector = LogisticRegressionCV(
    penalty="l1",
    solver="liblinear",
    Cs=np.logspace(-3, 1, 10),
    cv=cv,
    scoring="f1",
    class_weight="balanced",
    max_iter=5000,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
l1_selector.fit(X_train_linear, y_train)
lr_selected = list(X_train_linear.columns[np.abs(l1_selector.coef_).ravel() > 1e-8])
if not lr_selected:
    raise RuntimeError("L1/Lasso Logistic Regression selected zero features; stopping rather than benchmarking full features.")
selected_features["logistic_regression"] = lr_selected
feature_selection_details.append(
    {
        "model": "Logistic Regression",
        "method": FEATURE_SELECTION_METHODS["logistic_regression"],
        "selected_count": len(lr_selected),
        "details": f"Best C={float(l1_selector.C_[0]):.5g}",
    }
)

linear_svm_rfecv = RFECV(
    estimator=LinearSVC(penalty="l2", class_weight="balanced", max_iter=30000, dual="auto", random_state=RANDOM_STATE),
    step=0.1,
    min_features_to_select=min(5, X_train_linear.shape[1]),
    cv=cv,
    scoring="f1",
    n_jobs=1,
)
linear_svm_rfecv.fit(X_train_linear, y_train)
svm_selected = list(X_train_linear.columns[linear_svm_rfecv.support_])
if not svm_selected:
    raise RuntimeError("RFECV Linear SVM selected zero features; stopping rather than benchmarking full features.")
selected_features["linear_svm"] = svm_selected
feature_selection_details.append(
    {
        "model": "Linear SVM",
        "method": FEATURE_SELECTION_METHODS["linear_svm"],
        "selected_count": len(svm_selected),
        "details": f"RFECV selected {linear_svm_rfecv.n_features_} features",
    }
)

rf_for_boruta = RandomForestClassifier(n_estimators=1000, max_depth=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1)
boruta_selector = BorutaPy(
    estimator=rf_for_boruta,
    n_estimators="auto",
    perc=100,
    alpha=0.05,
    two_step=True,
    max_iter=100,
    random_state=RANDOM_STATE,
    verbose=0,
)
boruta_selector.fit(X_train_tree.values, y_train.values)
rf_selected_confirmed = list(X_train_tree.columns[boruta_selector.support_])
rf_selected_tentative = list(X_train_tree.columns[boruta_selector.support_weak_])
if not rf_selected_confirmed:
    raise RuntimeError(
        "Boruta selected zero confirmed Random Forest features; stopping rather than benchmarking Random Forest with full features. "
        f"Tentative features: {rf_selected_tentative}"
    )
selected_features["random_forest"] = rf_selected_confirmed
feature_selection_details.append(
    {
        "model": "Random Forest",
        "method": FEATURE_SELECTION_METHODS["random_forest"],
        "selected_count": len(rf_selected_confirmed),
        "details": f"Tentative features: {len(rf_selected_tentative)}",
    }
)

borutashap_catboost_model = CatBoostClassifier(
    iterations=BORUTASHAP_CATBOOST_ITERATIONS,
    depth=4,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="F1",
    auto_class_weights="Balanced",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)

X_train_borutashap = X_train_catboost.reset_index(drop=True).copy()
for col in catboost_categorical_features:
    if col in X_train_borutashap.columns:
        X_train_borutashap[col] = X_train_borutashap[col].astype("category")
y_train_borutashap = y_train.reset_index(drop=True).astype(int)

borutashap_selector = BorutaShap(
    model=borutashap_catboost_model,
    importance_measure="shap",
    classification=True,
    percentile=BORUTASHAP_PERCENTILE,
    pvalue=BORUTASHAP_PVALUE,
)
borutashap_selector.fit(
    X=X_train_borutashap,
    y=y_train_borutashap,
    n_trials=BORUTASHAP_N_TRIALS,
    random_state=RANDOM_STATE,
    sample=False,
    train_or_test="train",
    normalize=True,
    verbose=False,
)

borutashap_accepted_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.accepted)]
borutashap_tentative_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.tentative)]
borutashap_rejected_features = [col for col in X_train_catboost.columns if col in set(borutashap_selector.rejected)]
catboost_selected_features = borutashap_accepted_features.copy()
if not catboost_selected_features:
    raise RuntimeError("BorutaSHAP selected zero accepted CatBoost features; tentative features are reported but not auto-included.")

selected_features["catboost"] = catboost_selected_features
selected_catboost_categorical_features = [col for col in catboost_categorical_features if col in catboost_selected_features]

borutashap_importance = borutashap_selector.history_x.drop(columns=["Max_Shadow", "Min_Shadow", "Mean_Shadow", "Median_Shadow"], errors="ignore")
borutashap_importance_summary = pd.DataFrame(
    {
        "feature": list(borutashap_importance.columns),
        "mean_importance": borutashap_importance.mean(axis=0).values,
        "median_importance": borutashap_importance.median(axis=0).values,
        "max_importance": borutashap_importance.max(axis=0).values,
    }
)
borutashap_status_map = {
    **{feature: "accepted" for feature in borutashap_accepted_features},
    **{feature: "tentative" for feature in borutashap_tentative_features},
    **{feature: "rejected" for feature in borutashap_rejected_features},
}
borutashap_feature_status = pd.DataFrame(
    [{"feature": feature, "borutashap_status": borutashap_status_map.get(feature, "unassigned")} for feature in X_train_catboost.columns]
).merge(borutashap_importance_summary, on="feature", how="left")
borutashap_feature_status["used_in_main_catboost"] = borutashap_feature_status["feature"].isin(catboost_selected_features)
borutashap_feature_status = borutashap_feature_status.sort_values(
    ["used_in_main_catboost", "median_importance"], ascending=[False, False]
).reset_index(drop=True)

catboost_final_selected_features_table = pd.DataFrame({"selected_feature": catboost_selected_features})
borutashap_tentative_features_table = pd.DataFrame({"tentative_feature": borutashap_tentative_features})
borutashap_rejected_features_table = pd.DataFrame({"rejected_feature": borutashap_rejected_features})

feature_selection_details.append(
    {
        "model": "CatBoost",
        "method": FEATURE_SELECTION_METHODS["catboost"],
        "selected_count": len(catboost_selected_features),
        "details": f"Accepted={len(borutashap_accepted_features)}; tentative={len(borutashap_tentative_features)}; rejected={len(borutashap_rejected_features)}; selected categorical={len(selected_catboost_categorical_features)}",
    }
)

print(f"BorutaSHAP accepted CatBoost features: {len(borutashap_accepted_features)}")
print(f"BorutaSHAP tentative CatBoost features: {len(borutashap_tentative_features)}")
print(f"BorutaSHAP rejected CatBoost features: {len(borutashap_rejected_features)}")
display(borutashap_feature_status)
display(catboost_final_selected_features_table)
display(borutashap_tentative_features_table)
display(borutashap_rejected_features_table)

selected_feature_counts = {key: len(value) for key, value in selected_features.items()}
feature_selection_summary = pd.DataFrame(feature_selection_details)
selected_feature_table = pd.DataFrame(
    [{"model": model, "selected_feature": feature} for model, features in selected_features.items() for feature in features]
)

display(feature_selection_summary)
display(selected_feature_table)
display(borutashap_feature_status.head(15))

# 3. Model Benchmarking

This section benchmarks selected-feature models using training-only cross-validation. The held-out test set remains reserved for the final frozen model evaluation.

## 3.1 Evaluation Metrics Definition

F1-score and abnormal EEG recall/sensitivity are primary metrics. Accuracy is reported only as a secondary metric.

In [ ]:
def binary_metrics(y_true, y_score, threshold=0.5):
    y_true = pd.Series(y_true).astype(int).to_numpy()
    y_score = np.asarray(y_score, dtype=float)
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    try:
        roc_auc = roc_auc_score(y_true, y_score)
    except ValueError:
        roc_auc = np.nan
    return {
        "threshold": float(threshold),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def selected_matrix(model_key, split):
    if model_key in ["logistic_regression", "linear_svm"]:
        base = X_train_linear if split == "train" else X_test_linear
    elif model_key == "random_forest":
        base = X_train_tree if split == "train" else X_test_tree
    elif model_key == "catboost":
        base = X_train_catboost if split == "train" else X_test_catboost
    else:
        raise KeyError(model_key)
    return base[selected_features[model_key]].copy()


def make_model(model_key, params=None):
    params = params or {}
    if model_key == "logistic_regression":
        return LogisticRegression(C=params.get("C", 1.0), solver="liblinear", class_weight="balanced", max_iter=5000, random_state=RANDOM_STATE)
    if model_key == "linear_svm":
        base_svm = LinearSVC(C=params.get("C", 1.0), penalty="l2", class_weight="balanced", max_iter=30000, dual="auto", random_state=RANDOM_STATE)
        return CalibratedClassifierCV(estimator=base_svm, cv=3)
    if model_key == "random_forest":
        return RandomForestClassifier(
            n_estimators=params.get("n_estimators", 500),
            max_depth=params.get("max_depth", None),
            min_samples_leaf=params.get("min_samples_leaf", 1),
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
    if model_key == "catboost":
        return CatBoostClassifier(
            iterations=params.get("iterations", 200),
            depth=params.get("depth", 4),
            learning_rate=params.get("learning_rate", 0.05),
            l2_leaf_reg=params.get("l2_leaf_reg", 3),
            loss_function="Logloss",
            eval_metric="F1",
            auto_class_weights="Balanced",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )
    raise KeyError(model_key)


def cat_features_for_matrix(model_key, X_matrix):
    if model_key != "catboost":
        return []
    return [col for col in selected_catboost_categorical_features if col in X_matrix.columns]


def fit_model(model, model_key, X_fit, y_fit):
    if model_key == "catboost":
        model.fit(X_fit, y_fit, cat_features=cat_features_for_matrix(model_key, X_fit), verbose=False)
    else:
        model.fit(X_fit, y_fit)
    return model


def predict_scores(model, X_score):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X_score))[:, 1]
    decision = np.asarray(model.decision_function(X_score), dtype=float)
    return 1.0 / (1.0 + np.exp(-decision))


def oof_scores(model_key, X_matrix, y_values, params=None):
    y_values = pd.Series(y_values).astype(int)
    scores = pd.Series(index=y_values.index, dtype=float)
    for train_idx, valid_idx in cv.split(X_matrix, y_values):
        model = make_model(model_key, params=params)
        fit_model(model, model_key, X_matrix.iloc[train_idx].copy(), y_values.iloc[train_idx].copy())
        scores.iloc[valid_idx] = predict_scores(model, X_matrix.iloc[valid_idx].copy())
    return scores.astype(float)


metric_priority = ["f1", "recall_sensitivity", "balanced_accuracy"]
print("Metric helpers and model factories are ready.")

## 3.2 Modelling

The benchmark compares only selected-feature models using out-of-fold training predictions.

In [ ]:
MODEL_DISPLAY_NAMES = {
    "logistic_regression": "Logistic Regression",
    "linear_svm": "Linear SVM",
    "random_forest": "Random Forest",
    "catboost": "CatBoost",
}

benchmark_rows = []
benchmark_oof_scores = {}
for model_key in ["logistic_regression", "linear_svm", "random_forest", "catboost"]:
    X_model_train = selected_matrix(model_key, "train")
    scores = oof_scores(model_key, X_model_train, y_train)
    benchmark_oof_scores[model_key] = scores
    metrics = binary_metrics(y_train, scores, threshold=0.5)
    metrics.update(
        {
            "model_key": model_key,
            "model": MODEL_DISPLAY_NAMES[model_key],
            "feature_selection_method": FEATURE_SELECTION_METHODS[model_key],
            "selected_feature_count": len(selected_features[model_key]),
        }
    )
    benchmark_rows.append(metrics)

benchmark_results = pd.DataFrame(benchmark_rows)[
    [
        "model_key",
        "model",
        "selected_feature_count",
        "f1",
        "recall_sensitivity",
        "precision",
        "specificity",
        "balanced_accuracy",
        "accuracy",
        "roc_auc",
        "tn",
        "fp",
        "fn",
        "tp",
        "feature_selection_method",
    ]
].sort_values(metric_priority, ascending=False)

FINAL_MODEL_KEY = "catboost"
FINAL_MODEL_NAME = MODEL_DISPLAY_NAMES[FINAL_MODEL_KEY]
FINAL_MODEL_SELECTION_REASON = "CatBoost retained as final model because BorutaSHAP all-relevant ML risk-factor discovery is the project focus."

display(benchmark_results)
print(f"Selected final model for train-only tuning: {FINAL_MODEL_NAME}")
print(FINAL_MODEL_SELECTION_REASON)

# 4. Final Model Tuning

This section tunes the selected final model on training data only, tunes the threshold with training out-of-fold predictions only, and then evaluates the frozen model once on the held-out test set.

## 4.1 Final Model Tuning

The final model is tuned after feature selection using stratified cross-validation on the training set.

In [ ]:
TUNING_GRIDS = {
    "logistic_regression": {"C": [0.1, 1.0, 10.0]},
    "linear_svm": {"C": [0.1, 1.0, 10.0]},
    "random_forest": {"n_estimators": [500], "max_depth": [None, 5], "min_samples_leaf": [1, 3]},
    "catboost": {"iterations": [150, 250], "depth": [3, 4], "learning_rate": [0.03], "l2_leaf_reg": [3]},
}

X_train_final = selected_matrix(FINAL_MODEL_KEY, "train")
X_test_final = selected_matrix(FINAL_MODEL_KEY, "test")

tuning_rows = []
tuning_oof_by_params = []
for params in ParameterGrid(TUNING_GRIDS[FINAL_MODEL_KEY]):
    scores = oof_scores(FINAL_MODEL_KEY, X_train_final, y_train, params=params)
    metrics = binary_metrics(y_train, scores, threshold=0.5)
    row = {"model_key": FINAL_MODEL_KEY, "model": FINAL_MODEL_NAME, "params": params}
    row.update(metrics)
    tuning_rows.append(row)
    tuning_oof_by_params.append((params, scores, metrics))

tuning_results = pd.DataFrame(tuning_rows).sort_values(metric_priority, ascending=False).reset_index(drop=True)
best_tuning_params = tuning_results.iloc[0]["params"]
best_tuning_scores = next(scores for params, scores, _ in tuning_oof_by_params if params == best_tuning_params)

best_final_model = make_model(FINAL_MODEL_KEY, params=best_tuning_params)
fit_model(best_final_model, FINAL_MODEL_KEY, X_train_final, y_train)

print(f"Final model selected for tuning: {FINAL_MODEL_NAME}")
print(f"Best train-CV tuning parameters: {best_tuning_params}")
display(tuning_results)

## 4.2 Threshold Tuning

The classification threshold is selected with out-of-fold training predictions only. The held-out test labels are not used for threshold selection.

In [ ]:
def threshold_table(y_true, y_score, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.01), 2)
    return pd.DataFrame([binary_metrics(y_true, y_score, threshold=threshold) for threshold in thresholds])


threshold_results = threshold_table(y_train, best_tuning_scores)
threshold_results = threshold_results.sort_values(metric_priority, ascending=False).reset_index(drop=True)
selected_threshold = float(threshold_results.iloc[0]["threshold"])
threshold_selection_metrics = threshold_results.iloc[0].to_dict()

print(f"Selected threshold from training OOF predictions only: {selected_threshold:.2f}")
display(threshold_results.head(15))

## 4.3 Final Held-Out Test Evaluation

The tuned model and selected threshold are frozen here, then evaluated once on the held-out test set.

In [ ]:
test_scores = predict_scores(best_final_model, X_test_final)
final_test_metrics = binary_metrics(y_test, test_scores, threshold=selected_threshold)
final_test_results = pd.DataFrame([{**{"model": FINAL_MODEL_NAME, "params": best_tuning_params}, **final_test_metrics}])

previous_catboost_reference = {
    "selected_feature_count": 10,
    "threshold": 0.30,
    "f1": 0.835,
    "recall_sensitivity": 0.905,
    "precision": 0.776,
    "specificity": 0.312,
    "balanced_accuracy": 0.609,
    "accuracy": 0.741,
    "roc_auc": 0.766,
    "tn": 5,
    "fp": 11,
    "fn": 4,
    "tp": 38,
}
current_catboost_summary = {
    "selected_feature_count": len(catboost_selected_features),
    **final_test_metrics,
}
catboost_metric_changes_vs_previous = pd.DataFrame(
    [
        {
            "metric": metric,
            "previous_catboost": previous_value,
            "current_catboost": current_catboost_summary.get(metric, selected_threshold if metric == "threshold" else np.nan),
            "change": current_catboost_summary.get(metric, selected_threshold if metric == "threshold" else np.nan) - previous_value,
        }
        for metric, previous_value in previous_catboost_reference.items()
    ]
)
final_confusion_matrix = pd.DataFrame(
    [[final_test_metrics["tn"], final_test_metrics["fp"]], [final_test_metrics["fn"], final_test_metrics["tp"]]],
    index=["Actual normal", "Actual abnormal"],
    columns=["Predicted normal", "Predicted abnormal"],
)

print("Final held-out test evaluation complete. This is the first metric evaluation using held-out test labels.")
display(final_test_results)
display(final_confusion_matrix)
print("Metric changes versus previous CatBoost reference result:")
display(catboost_metric_changes_vs_previous)

final_test_results.to_csv(OUTPUT_DIR / "final_held_out_test_metrics.csv", index=False)
benchmark_results.to_csv(OUTPUT_DIR / "benchmark_results_train_cv.csv", index=False)
feature_selection_summary.to_csv(OUTPUT_DIR / "feature_selection_summary.csv", index=False)
selected_feature_table.to_csv(OUTPUT_DIR / "selected_features.csv", index=False)
borutashap_feature_status.to_csv(OUTPUT_DIR / "catboost_borutashap_feature_status.csv", index=False)
catboost_final_selected_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_accepted_features.csv", index=False)
borutashap_tentative_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_tentative_features.csv", index=False)
borutashap_rejected_features_table.to_csv(OUTPUT_DIR / "catboost_borutashap_rejected_features.csv", index=False)
catboost_metric_changes_vs_previous.to_csv(OUTPUT_DIR / "catboost_metric_changes_vs_previous.csv", index=False)

# 5. Explainable AI

This section explains and triangulates the final model using Standard SHAP, dependence-aware Conditional SHAP, EBM, and exploratory DML. These methods explain model behavior or adjusted associations; they do not establish causality, clinical cut-offs, diagnostic validity, or deployment readiness.


In [ ]:
from pathlib import Path
import inspect
import json
import platform
import sys

import catboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, CatBoostRegressor, EFstrType, Pool
from scipy.stats import kendalltau, spearmanr

XAI_OUTPUT_DIR = OUTPUT_DIR / "explainable_ai"
XAI_TABLES_DIR = XAI_OUTPUT_DIR / "tables"
STANDARD_SHAP_DIR = XAI_OUTPUT_DIR / "standard_shap"
CONDITIONAL_SHAP_DIR = XAI_OUTPUT_DIR / "conditional_shap"
COMPARISON_DIR = XAI_OUTPUT_DIR / "shap_comparison"
EBM_OUTPUT_DIR = XAI_OUTPUT_DIR / "ebm"
DML_OUTPUT_DIR = XAI_OUTPUT_DIR / "dml"

for directory in (
    XAI_OUTPUT_DIR,
    XAI_TABLES_DIR,
    STANDARD_SHAP_DIR,
    CONDITIONAL_SHAP_DIR,
    COMPARISON_DIR,
    EBM_OUTPUT_DIR,
    DML_OUTPUT_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

XAI_TOP_FEATURE_COUNT = min(5, X_test_final.shape[1])
XAI_COMPARISON_TOP_K = min(3, X_test_final.shape[1])


def safe_feature_filename(feature_name):
    return "".join(
        character if character.isalnum() else "_"
        for character in str(feature_name)
    ).strip("_")[:100]


def inverse_numeric_to_clinical(feature, values):
    values = np.asarray(values, dtype=float)
    if feature in clinical_preprocessor.log_transform_columns_:
        return np.expm1(values)
    return values


def training_category_levels(X_train_frame, categorical_features):
    return {
        feature: list(pd.unique(X_train_frame[feature].astype(str)))
        for feature in categorical_features
    }


def xai_display_frame(X_frame, categorical_features, category_levels):
    display_frame = pd.DataFrame(index=X_frame.index)
    category_labels = {}
    for feature in X_frame.columns:
        if feature in categorical_features:
            levels = list(category_levels[feature])
            categorical = pd.Categorical(
                X_frame[feature].astype(str), categories=levels, ordered=False
            )
            if np.any(categorical.codes < 0):
                raise ValueError(
                    f"Explanation feature '{feature}' contains a category absent "
                    "from training data."
                )
            display_frame[feature] = categorical.codes.astype(float)
            category_labels[feature] = levels
        else:
            numeric = pd.to_numeric(X_frame[feature], errors="raise").to_numpy(float)
            display_frame[feature] = inverse_numeric_to_clinical(feature, numeric)
    return display_frame.astype(float), category_labels


def plot_xai_dependence(
    feature,
    shap_matrix,
    display_X,
    category_labels,
    output_path,
    title_prefix,
):
    feature_index = display_X.columns.get_loc(feature)
    x_values = display_X[feature].to_numpy(dtype=float)
    y_values = np.asarray(shap_matrix, dtype=float)[:, feature_index]
    rng = np.random.default_rng(RANDOM_STATE)

    if feature in category_labels:
        x_plot = x_values + rng.normal(0, 0.035, len(x_values))
    else:
        x_plot = x_values

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(
        x_plot,
        y_values,
        s=44,
        alpha=0.78,
        color="#E8753D",
        edgecolor="#C85A28",
        linewidth=0.6,
    )
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_xlabel(feature)
    ax.set_ylabel(f"{title_prefix} SHAP value (raw margin)")
    ax.set_title(f"{title_prefix} SHAP Dependence: {feature}")
    if feature in category_labels:
        labels = category_labels[feature]
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(output_path, dpi=250, bbox_inches="tight")
    plt.close(fig)


selected_final_features = list(X_train_final.columns)
selected_final_categorical_features = [
    feature
    for feature in selected_catboost_categorical_features
    if feature in selected_final_features
]

if selected_final_features != list(X_test_final.columns):
    raise RuntimeError("Final train/test feature order differs before XAI execution.")
if set(selected_final_features) != set(catboost_selected_features):
    raise RuntimeError("Final XAI features differ from the frozen CatBoost feature set.")
if set(selected_final_features).intersection(LEAKAGE_COLUMNS):
    raise RuntimeError("Leakage feature detected in the final XAI feature set.")

xai_category_levels = training_category_levels(
    X_train_final, selected_final_categorical_features
)
standard_shap_display_X, standard_category_labels = xai_display_frame(
    X_test_final,
    selected_final_categorical_features,
    xai_category_levels,
)

xai_configuration_df = pd.DataFrame(
    [
        {
            "Final_Model": FINAL_MODEL_NAME,
            "Output_Scale": "CatBoost RawFormulaVal",
            "Training_Observation_Count": len(X_train_final),
            "Explanation_Observation_Count": len(X_test_final),
            "Selected_Feature_Count": len(selected_final_features),
            "Selected_Features": json.dumps(selected_final_features),
            "Categorical_Features": json.dumps(selected_final_categorical_features),
            "Random_State": RANDOM_STATE,
        }
    ]
)
xai_configuration_df.to_csv(
    XAI_TABLES_DIR / "xai_configuration.csv", index=False
)

print("Explainable AI configuration")
display(xai_configuration_df)


## 5.1 SHAP Analysis

Use Standard SHAP as the conventional explanation baseline for the frozen final model. Do not use held-out SHAP values for feature selection, model tuning, or threshold tuning.


### 5.1.1 SHAP Execution

Create model-appropriate SHAP explanations on a clearly defined explanation dataset. Preserve feature names, feature order, observation indices, and the model-output scale.


In [ ]:
standard_shap_pool = Pool(
    X_test_final,
    cat_features=selected_final_categorical_features,
)
standard_shap_with_baseline = best_final_model.get_feature_importance(
    standard_shap_pool,
    type=EFstrType.ShapValues,
)
standard_shap_with_baseline = np.asarray(
    standard_shap_with_baseline, dtype=float
)
standard_shap_values = standard_shap_with_baseline[:, :-1]
standard_shap_base_values = standard_shap_with_baseline[:, -1]
standard_raw_margin_predictions = np.asarray(
    best_final_model.predict(
        X_test_final,
        prediction_type="RawFormulaVal",
    ),
    dtype=float,
).reshape(-1)

expected_standard_shape = (len(X_test_final), len(selected_final_features))
if standard_shap_values.shape != expected_standard_shape:
    raise RuntimeError(
        f"Standard SHAP shape {standard_shap_values.shape} does not match "
        f"{expected_standard_shape}."
    )
if not np.isfinite(standard_shap_values).all():
    raise RuntimeError("Standard SHAP contains non-finite values.")

standard_reconstructed_margin = (
    standard_shap_base_values + standard_shap_values.sum(axis=1)
)
standard_additivity_residual = (
    standard_raw_margin_predictions - standard_reconstructed_margin
)
np.testing.assert_allclose(
    standard_reconstructed_margin,
    standard_raw_margin_predictions,
    rtol=1e-8,
    atol=1e-6,
)

standard_shap_explanation = shap.Explanation(
    values=standard_shap_values,
    base_values=standard_shap_base_values,
    data=standard_shap_display_X.to_numpy(dtype=float),
    feature_names=selected_final_features,
)

standard_shap_local_values_df = pd.DataFrame(
    standard_shap_values,
    columns=selected_final_features,
)
standard_shap_local_values_df.insert(
    0, "Explanation_Position", np.arange(len(standard_shap_local_values_df))
)
standard_shap_local_values_df["Base_Value"] = standard_shap_base_values
standard_shap_local_values_df["Raw_Margin"] = standard_raw_margin_predictions
standard_shap_local_values_df["Additivity_Residual"] = standard_additivity_residual
standard_shap_local_values_df.to_csv(
    XAI_TABLES_DIR / "standard_shap_local_values.csv", index=False
)

standard_shap_validation_df = pd.DataFrame(
    [
        {
            "Expected_Shape": str(expected_standard_shape),
            "Observed_Shape": str(standard_shap_values.shape),
            "Finite_Values": bool(np.isfinite(standard_shap_values).all()),
            "Maximum_Absolute_Additivity_Residual": float(
                np.abs(standard_additivity_residual).max()
            ),
            "Additivity_Passed": True,
        }
    ]
)
standard_shap_validation_df.to_csv(
    XAI_TABLES_DIR / "standard_shap_validation.csv", index=False
)

standard_mean_abs_shap = np.abs(standard_shap_values).mean(axis=0)
standard_shap_ranking_df = (
    pd.DataFrame(
        {
            "Feature": selected_final_features,
            "MeanAbsSHAP": standard_mean_abs_shap,
        }
    )
    .sort_values("MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)
standard_shap_ranking_df["Standard_SHAP_Rank"] = np.arange(
    1, len(standard_shap_ranking_df) + 1
)
standard_shap_ranking_df.to_csv(
    XAI_TABLES_DIR / "standard_shap_ranking.csv", index=False
)
standard_top_features = standard_shap_ranking_df.head(
    XAI_TOP_FEATURE_COUNT
)["Feature"].tolist()

print("Standard SHAP validation")
display(standard_shap_validation_df)
print("Standard SHAP ranking")
display(standard_shap_ranking_df)


### 5.1.2 Global SHAP Bar Plot

Rank the final model predictors using mean absolute Standard SHAP values.


In [ ]:
shap.plots.bar(
    standard_shap_explanation,
    max_display=len(selected_final_features),
    show=False,
)
plt.tight_layout()
standard_shap_bar_path = STANDARD_SHAP_DIR / "standard_shap_global_bar.png"
plt.savefig(standard_shap_bar_path, dpi=250, bbox_inches="tight")
plt.close()
print(f"Saved: {standard_shap_bar_path}")


### 5.1.3 SHAP Beeswarm Plot

Show the distribution, direction, and relative magnitude of Standard SHAP values across explained observations.


In [ ]:
shap.plots.beeswarm(
    standard_shap_explanation,
    max_display=len(selected_final_features),
    show=False,
)
plt.tight_layout()
standard_shap_beeswarm_path = (
    STANDARD_SHAP_DIR / "standard_shap_beeswarm.png"
)
plt.savefig(standard_shap_beeswarm_path, dpi=250, bbox_inches="tight")
plt.close()
print(f"Saved: {standard_shap_beeswarm_path}")


### 5.1.4 SHAP Waterfall Plot

Present a local explanation for a non-identifying observation selected by a predefined, reproducible rule.


In [ ]:
median_raw_margin = float(np.median(standard_raw_margin_predictions))
local_explanation_position = int(
    np.argmin(np.abs(standard_raw_margin_predictions - median_raw_margin))
)
local_explanation_rule_df = pd.DataFrame(
    [
        {
            "Rule": "Held-out raw margin closest to the held-out median raw margin",
            "Explanation_Position": local_explanation_position,
            "Raw_Margin": float(
                standard_raw_margin_predictions[local_explanation_position]
            ),
        }
    ]
)
local_explanation_rule_df.to_csv(
    XAI_TABLES_DIR / "local_explanation_selection_rule.csv", index=False
)

shap.plots.waterfall(
    standard_shap_explanation[local_explanation_position],
    max_display=len(selected_final_features),
    show=False,
)
plt.tight_layout()
standard_shap_waterfall_path = (
    STANDARD_SHAP_DIR / "standard_shap_waterfall_predefined_case.png"
)
plt.savefig(standard_shap_waterfall_path, dpi=250, bbox_inches="tight")
plt.close()
print("Predefined non-identifying local explanation rule")
display(local_explanation_rule_df)
print(f"Saved: {standard_shap_waterfall_path}")


### 5.1.5 SHAP Dependence Plots for Important Features

Inspect how values of the most important predictors relate to their Standard SHAP contributions. Any apparent transition is exploratory and not a validated clinical cut-off.


In [ ]:
standard_dependence_plot_paths = []
for feature in standard_top_features:
    output_path = (
        STANDARD_SHAP_DIR
        / f"standard_shap_dependence_{safe_feature_filename(feature)}.png"
    )
    plot_xai_dependence(
        feature,
        standard_shap_values,
        standard_shap_display_X,
        standard_category_labels,
        output_path,
        "Standard",
    )
    standard_dependence_plot_paths.append(output_path)

print("Standard SHAP dependence plots")
for path in standard_dependence_plot_paths:
    print(path)


## 5.2 Conditional SHAP Analysis

Estimate dependence-aware Conditional SHAP values through Python `pyshapr`, R `shapr`, and conditional inference trees (`approach="ctree"`). Learn conditional feature distributions from training data only and explain a separately defined explanation dataset on the same model-output scale used by Standard SHAP.


### 5.2.1 Conditional SHAP Execution

Check dependencies, prepare numerical values and training-derived categorical factor levels, create safe internal feature aliases, define the final-model prediction wrapper, execute CTREE Conditional SHAP, and extract the attribution matrix and baseline.


In [ ]:
import pyshapr
from rpy2 import robjects as ro

CONDITIONAL_SHAP_SEED = RANDOM_STATE
CONDITIONAL_SHAP_MC_SAMPLES = 1000
CONDITIONAL_WRAPPER_ATOL = 1e-12
CONDITIONAL_ADDITIVITY_ATOL = 1e-6
CONDITIONAL_ADDITIVITY_RTOL = 1e-8

conditional_feature_names = list(selected_final_features)
conditional_categorical_features = list(selected_final_categorical_features)
conditional_numeric_features = [
    feature
    for feature in conditional_feature_names
    if feature not in conditional_categorical_features
]

conditional_alias_names = [
    f"eeg_xai_feature_{position + 1}"
    for position in range(len(conditional_feature_names))
]
conditional_original_to_alias = dict(
    zip(conditional_feature_names, conditional_alias_names)
)
conditional_alias_to_original = {
    alias: original
    for original, alias in conditional_original_to_alias.items()
}

conditional_train_X = X_train_final.loc[:, conditional_feature_names].copy()
conditional_explanation_X = X_test_final.loc[
    :, conditional_feature_names
].copy()

conditional_category_levels = {}
for feature in conditional_categorical_features:
    train_strings = conditional_train_X[feature].astype(str)
    explanation_strings = conditional_explanation_X[feature].astype(str)
    levels = list(pd.unique(train_strings))
    unseen_levels = sorted(set(explanation_strings) - set(levels))
    if unseen_levels:
        raise ValueError(
            f"Held-out feature '{feature}' contains unseen categories: "
            f"{unseen_levels}."
        )
    category_dtype = pd.CategoricalDtype(categories=levels, ordered=False)
    conditional_train_X[feature] = train_strings.astype(category_dtype)
    conditional_explanation_X[feature] = explanation_strings.astype(
        category_dtype
    )
    if (
        conditional_train_X[feature].isna().any()
        or conditional_explanation_X[feature].isna().any()
    ):
        raise ValueError(
            f"Categorical conversion introduced an invalid value in '{feature}'."
        )
    conditional_category_levels[feature] = levels

for feature in conditional_numeric_features:
    conditional_train_X[feature] = pd.to_numeric(
        conditional_train_X[feature], errors="raise"
    )
    conditional_explanation_X[feature] = pd.to_numeric(
        conditional_explanation_X[feature], errors="raise"
    )
    if not np.isfinite(conditional_train_X[feature].to_numpy(float)).all():
        raise ValueError(f"Training numeric feature '{feature}' is not finite.")
    if not np.isfinite(
        conditional_explanation_X[feature].to_numpy(float)
    ).all():
        raise ValueError(f"Held-out numeric feature '{feature}' is not finite.")

conditional_train_alias_X = (
    conditional_train_X.rename(columns=conditional_original_to_alias)
    .loc[:, conditional_alias_names]
    .copy()
)
conditional_explanation_alias_X = (
    conditional_explanation_X.rename(columns=conditional_original_to_alias)
    .loc[:, conditional_alias_names]
    .copy()
)


def catboost_raw_margin_predict(model, newdata):
    frame = pd.DataFrame(newdata, columns=conditional_alias_names).copy()
    frame = (
        frame.loc[:, conditional_alias_names]
        .rename(columns=conditional_alias_to_original)
        .loc[:, conditional_feature_names]
    )

    for feature in conditional_categorical_features:
        if frame[feature].isna().any():
            raise ValueError(
                f"Conditional sample contains an invalid category in '{feature}'."
            )
        sampled_levels = set(frame[feature].astype(str))
        unexpected_levels = sampled_levels - set(
            conditional_category_levels[feature]
        )
        if unexpected_levels:
            raise ValueError(
                f"Conditional sample for '{feature}' contains unexpected "
                f"levels: {sorted(unexpected_levels)}"
            )
        frame[feature] = frame[feature].astype(str)

    for feature in conditional_numeric_features:
        frame[feature] = pd.to_numeric(frame[feature], errors="raise")

    predictions = model.predict(frame, prediction_type="RawFormulaVal")
    return np.asarray(predictions, dtype=float).reshape(-1)


conditional_feature_classes = {
    conditional_original_to_alias[feature]: (
        "factor" if feature in conditional_categorical_features else "numeric"
    )
    for feature in conditional_feature_names
}


def get_catboost_model_specs(model):
    return {
        "labels": np.asarray(conditional_alias_names, dtype=str),
        "classes": conditional_feature_classes,
        "factor_levels": None,
    }


conditional_direct_raw_margin = np.asarray(
    best_final_model.predict(
        X_test_final.loc[:, conditional_feature_names],
        prediction_type="RawFormulaVal",
    ),
    dtype=float,
).reshape(-1)
conditional_wrapper_raw_margin = catboost_raw_margin_predict(
    best_final_model,
    conditional_explanation_alias_X,
)
conditional_train_raw_margin = catboost_raw_margin_predict(
    best_final_model,
    conditional_train_alias_X,
)
conditional_shap_baseline = float(conditional_train_raw_margin.mean())
conditional_coalition_count = 2 ** len(conditional_feature_names)

conditional_shap_result = pyshapr.explain(
    model=best_final_model,
    x_train=conditional_train_alias_X,
    x_explain=conditional_explanation_alias_X,
    approach="ctree",
    phi0=conditional_shap_baseline,
    iterative=False,
    max_n_coalitions=conditional_coalition_count,
    n_MC_samples=CONDITIONAL_SHAP_MC_SAMPLES,
    seed=CONDITIONAL_SHAP_SEED,
    verbose="basic",
    predict_model=catboost_raw_margin_predict,
    get_model_specs=get_catboost_model_specs,
)

conditional_shap_result_df = conditional_shap_result.get_results(
    "shapley_est"
).copy()
conditional_shap_values = conditional_shap_result_df[
    conditional_alias_names
].to_numpy(dtype=float)

print(
    "Conditional SHAP execution complete: "
    f"{len(conditional_explanation_X)} held-out observations, "
    f"{len(conditional_feature_names)} features, "
    f"{conditional_coalition_count} coalitions."
)


### 5.2.2 Conditional SHAP Validation

Validate prediction-wrapper equivalence, feature names and order, categorical levels, explained-row mapping, finite values, and attribution additivity. Save method, dependency, version, seed, and execution-environment metadata.


In [ ]:
expected_conditional_columns = [
    "explain_id",
    "none",
    *conditional_alias_names,
]
if list(conditional_shap_result_df.columns) != expected_conditional_columns:
    raise RuntimeError(
        "Conditional SHAP result columns do not match the expected order. "
        f"Expected {expected_conditional_columns}, got "
        f"{list(conditional_shap_result_df.columns)}."
    )

expected_explain_ids = np.arange(1, len(conditional_explanation_X) + 1)
if not np.array_equal(
    conditional_shap_result_df["explain_id"].to_numpy(),
    expected_explain_ids,
):
    raise RuntimeError("Conditional SHAP explain_id order is invalid.")

conditional_returned_explanation_X = conditional_shap_result.get_results(
    "x_explain"
).loc[:, conditional_alias_names].reset_index(drop=True)
conditional_expected_explanation_X = conditional_explanation_alias_X.reset_index(
    drop=True
)
for feature in conditional_feature_names:
    alias = conditional_original_to_alias[feature]
    if feature in conditional_categorical_features:
        if not np.array_equal(
            conditional_returned_explanation_X[alias].astype(str).to_numpy(),
            conditional_expected_explanation_X[alias].astype(str).to_numpy(),
        ):
            raise RuntimeError(
                f"Conditional SHAP row mapping failed for '{feature}'."
            )
    else:
        np.testing.assert_allclose(
            conditional_returned_explanation_X[alias].to_numpy(float),
            conditional_expected_explanation_X[alias].to_numpy(float),
            rtol=0.0,
            atol=0.0,
        )

expected_conditional_shape = (
    len(conditional_explanation_X),
    len(conditional_feature_names),
)
if conditional_shap_values.shape != expected_conditional_shape:
    raise RuntimeError(
        f"Conditional SHAP shape {conditional_shap_values.shape} does not "
        f"match {expected_conditional_shape}."
    )
if not np.isfinite(conditional_shap_values).all():
    raise RuntimeError("Conditional SHAP contains non-finite values.")

conditional_wrapper_difference = (
    conditional_wrapper_raw_margin - conditional_direct_raw_margin
)
np.testing.assert_allclose(
    conditional_wrapper_raw_margin,
    conditional_direct_raw_margin,
    rtol=0.0,
    atol=CONDITIONAL_WRAPPER_ATOL,
)

conditional_returned_baselines = conditional_shap_result_df[
    "none"
].to_numpy(dtype=float)
np.testing.assert_allclose(
    conditional_returned_baselines,
    conditional_shap_baseline,
    rtol=0.0,
    atol=CONDITIONAL_ADDITIVITY_ATOL,
)

conditional_returned_predictions = np.asarray(
    conditional_shap_result.get_results("pred_explain"),
    dtype=float,
).reshape(-1)
np.testing.assert_allclose(
    conditional_returned_predictions,
    conditional_direct_raw_margin,
    rtol=0.0,
    atol=CONDITIONAL_WRAPPER_ATOL,
)

conditional_reconstructed_raw_margin = (
    conditional_shap_baseline + conditional_shap_values.sum(axis=1)
)
conditional_additivity_residual = (
    conditional_direct_raw_margin - conditional_reconstructed_raw_margin
)
conditional_additivity_pass = np.isclose(
    conditional_reconstructed_raw_margin,
    conditional_direct_raw_margin,
    rtol=CONDITIONAL_ADDITIVITY_RTOL,
    atol=CONDITIONAL_ADDITIVITY_ATOL,
)
if not conditional_additivity_pass.all():
    raise RuntimeError(
        "Conditional SHAP additivity failed for "
        f"{int((~conditional_additivity_pass).sum())} observations."
    )

conditional_shap_additivity_df = pd.DataFrame(
    {
        "Explanation_Position": np.arange(len(conditional_explanation_X)),
        "Direct_CatBoost_RawFormulaVal": conditional_direct_raw_margin,
        "Conditional_SHAP_Reconstructed_RawFormulaVal": (
            conditional_reconstructed_raw_margin
        ),
        "Additivity_Residual": conditional_additivity_residual,
        "Absolute_Additivity_Residual": np.abs(
            conditional_additivity_residual
        ),
        "Within_Tolerance": conditional_additivity_pass,
    }
)
conditional_shap_additivity_df.to_csv(
    XAI_TABLES_DIR / "conditional_shap_additivity_check.csv", index=False
)

conditional_shap_local_values_df = pd.DataFrame(
    conditional_shap_values,
    columns=conditional_feature_names,
)
conditional_shap_local_values_df.insert(
    0, "Explanation_Position", np.arange(len(conditional_explanation_X))
)
conditional_shap_local_values_df["Conditional_SHAP_Baseline"] = (
    conditional_shap_baseline
)
conditional_shap_local_values_df["CatBoost_RawFormulaVal"] = (
    conditional_direct_raw_margin
)
conditional_shap_local_values_df["Reconstructed_RawFormulaVal"] = (
    conditional_reconstructed_raw_margin
)
conditional_shap_local_values_df["Additivity_Residual"] = (
    conditional_additivity_residual
)
conditional_shap_local_values_df.to_csv(
    XAI_TABLES_DIR / "conditional_shap_local_values.csv", index=False
)

r_version = str(ro.r("R.version.string")[0])
r_shapr_version = str(
    ro.r('as.character(packageVersion("shapr"))')[0]
)
conditional_shap_method_metadata_df = pd.DataFrame(
    [
        {
            "Method": "Dependence-aware Conditional SHAP",
            "Execution_Chain": "Python pyshapr -> R shapr -> CTREE",
            "Explained_Model": FINAL_MODEL_NAME,
            "Output_Scale": "CatBoost RawFormulaVal",
            "Selected_Features": json.dumps(conditional_feature_names),
            "Categorical_Features": json.dumps(
                conditional_categorical_features
            ),
            "Training_Derived_Category_Levels": json.dumps(
                conditional_category_levels
            ),
            "Feature_Aliases": json.dumps(conditional_original_to_alias),
            "Training_Observation_Count": len(conditional_train_X),
            "Explanation_Observation_Count": len(
                conditional_explanation_X
            ),
            "Coalition_Count": conditional_coalition_count,
            "Monte_Carlo_Sample_Count": CONDITIONAL_SHAP_MC_SAMPLES,
            "Seed": CONDITIONAL_SHAP_SEED,
            "Python_Version": platform.python_version(),
            "R_Version": r_version,
            "pyshapr_Version": pyshapr.__version__,
            "shapr_Version": r_shapr_version,
            "CatBoost_Version": catboost.__version__,
            "Operating_System": platform.platform(),
            "Python_Environment": sys.prefix,
            "pyshapr_Explain_Signature": str(
                inspect.signature(pyshapr._explain_impl)
            ),
        }
    ]
)
conditional_shap_method_metadata_df.to_csv(
    XAI_TABLES_DIR / "conditional_shap_method_metadata.csv", index=False
)

conditional_shap_validation_df = pd.DataFrame(
    [
        {
            "Wrapper_Equivalence_Passed": True,
            "Wrapper_Maximum_Absolute_Difference": float(
                np.abs(conditional_wrapper_difference).max()
            ),
            "Feature_Order_Passed": True,
            "Category_Level_Validation_Passed": True,
            "Explanation_Row_Mapping_Passed": True,
            "Finite_Values_Passed": bool(
                np.isfinite(conditional_shap_values).all()
            ),
            "Additivity_Passed": bool(conditional_additivity_pass.all()),
            "Additivity_Maximum_Absolute_Residual": float(
                np.abs(conditional_additivity_residual).max()
            ),
            "Additivity_Mean_Absolute_Residual": float(
                np.abs(conditional_additivity_residual).mean()
            ),
        }
    ]
)
conditional_shap_validation_df.to_csv(
    XAI_TABLES_DIR / "conditional_shap_validation.csv", index=False
)

conditional_shap_explanation = shap.Explanation(
    values=conditional_shap_values,
    base_values=np.repeat(
        conditional_shap_baseline, len(conditional_explanation_X)
    ),
    data=standard_shap_display_X.to_numpy(dtype=float),
    feature_names=conditional_feature_names,
)

print("Conditional SHAP validation")
display(conditional_shap_validation_df)
print("Conditional SHAP method metadata")
display(conditional_shap_method_metadata_df)


### 5.2.3 Conditional SHAP Global Bar Plot

Rank predictors using mean absolute per-observation Conditional SHAP values. Do not label this aggregation as SAGE unless a separate global-scope SAGE analysis is performed.


In [ ]:
conditional_mean_abs_shap = np.abs(conditional_shap_values).mean(axis=0)
conditional_shap_ranking_df = (
    pd.DataFrame(
        {
            "Feature": conditional_feature_names,
            "MeanAbsConditionalSHAP": conditional_mean_abs_shap,
        }
    )
    .sort_values("MeanAbsConditionalSHAP", ascending=False)
    .reset_index(drop=True)
)
conditional_shap_ranking_df["Conditional_SHAP_Rank"] = np.arange(
    1, len(conditional_shap_ranking_df) + 1
)
conditional_shap_ranking_df.to_csv(
    XAI_TABLES_DIR / "conditional_shap_ranking.csv", index=False
)
conditional_top_features = conditional_shap_ranking_df.head(
    XAI_TOP_FEATURE_COUNT
)["Feature"].tolist()

shap.plots.bar(
    conditional_shap_explanation,
    max_display=len(conditional_feature_names),
    show=False,
)
plt.tight_layout()
conditional_shap_bar_path = (
    CONDITIONAL_SHAP_DIR / "conditional_shap_global_bar.png"
)
plt.savefig(conditional_shap_bar_path, dpi=250, bbox_inches="tight")
plt.close()
display(conditional_shap_ranking_df)
print(f"Saved: {conditional_shap_bar_path}")


### 5.2.4 Conditional SHAP Beeswarm Plot

Show the distribution, direction, and relative magnitude of dependence-aware Conditional SHAP values.


In [ ]:
shap.plots.beeswarm(
    conditional_shap_explanation,
    max_display=len(conditional_feature_names),
    show=False,
)
plt.tight_layout()
conditional_shap_beeswarm_path = (
    CONDITIONAL_SHAP_DIR / "conditional_shap_beeswarm.png"
)
plt.savefig(conditional_shap_beeswarm_path, dpi=250, bbox_inches="tight")
plt.close()
print(f"Saved: {conditional_shap_beeswarm_path}")


### 5.2.5 Conditional SHAP Waterfall Plot

Present a local Conditional SHAP explanation using the validated conditional baseline and the same predefined, non-identifying observation rule used for Standard SHAP.


In [ ]:
shap.plots.waterfall(
    conditional_shap_explanation[local_explanation_position],
    max_display=len(conditional_feature_names),
    show=False,
)
plt.tight_layout()
conditional_shap_waterfall_path = (
    CONDITIONAL_SHAP_DIR / "conditional_shap_waterfall_predefined_case.png"
)
plt.savefig(conditional_shap_waterfall_path, dpi=250, bbox_inches="tight")
plt.close()
print(f"Saved: {conditional_shap_waterfall_path}")


### 5.2.6 Conditional SHAP Dependence Plots for Important Features

Inspect important feature values against their dependence-aware Conditional SHAP contributions. Apparent transitions remain exploratory.


In [ ]:
conditional_dependence_plot_paths = []
for feature in conditional_top_features:
    output_path = (
        CONDITIONAL_SHAP_DIR
        / f"conditional_shap_dependence_{safe_feature_filename(feature)}.png"
    )
    plot_xai_dependence(
        feature,
        conditional_shap_values,
        standard_shap_display_X,
        standard_category_labels,
        output_path,
        "Conditional",
    )
    conditional_dependence_plot_paths.append(output_path)

print("Conditional SHAP dependence plots")
for path in conditional_dependence_plot_paths:
    print(path)


## 5.3 Standard vs Conditional SHAP Comparison

Compare Standard and Conditional SHAP only when both explain the same frozen model, features, observations, and model-output scale. Evaluate feature ranks, attribution magnitudes, rank correlations, and top-feature overlap. This is an explanation-robustness analysis, not causal validation.


In [ ]:
shap_comparison_df = standard_shap_ranking_df.merge(
    conditional_shap_ranking_df,
    on="Feature",
    how="inner",
    validate="one_to_one",
)
if len(shap_comparison_df) != len(selected_final_features):
    raise RuntimeError(
        "Standard/Conditional SHAP comparison does not cover every final feature."
    )

standard_importance_total = float(shap_comparison_df["MeanAbsSHAP"].sum())
conditional_importance_total = float(
    shap_comparison_df["MeanAbsConditionalSHAP"].sum()
)
shap_comparison_df["Standard_Normalized_Importance"] = (
    shap_comparison_df["MeanAbsSHAP"] / standard_importance_total
)
shap_comparison_df["Conditional_Normalized_Importance"] = (
    shap_comparison_df["MeanAbsConditionalSHAP"]
    / conditional_importance_total
)
shap_comparison_df["Normalized_Importance_Change"] = (
    shap_comparison_df["Conditional_Normalized_Importance"]
    - shap_comparison_df["Standard_Normalized_Importance"]
)
shap_comparison_df["Rank_Change"] = (
    shap_comparison_df["Conditional_SHAP_Rank"]
    - shap_comparison_df["Standard_SHAP_Rank"]
)
shap_comparison_df["Absolute_Rank_Change"] = (
    shap_comparison_df["Rank_Change"].abs().astype(int)
)
shap_comparison_df["Interpretation"] = np.select(
    [
        shap_comparison_df["Absolute_Rank_Change"].eq(0),
        shap_comparison_df["Absolute_Rank_Change"].eq(1),
    ],
    ["Stable", "Minor rank shift"],
    default="Changed",
)
shap_comparison_df = shap_comparison_df.sort_values(
    "Standard_SHAP_Rank"
).reset_index(drop=True)

spearman_rank_correlation = float(
    spearmanr(
        shap_comparison_df["Standard_SHAP_Rank"],
        shap_comparison_df["Conditional_SHAP_Rank"],
    ).statistic
)
kendall_rank_correlation = float(
    kendalltau(
        shap_comparison_df["Standard_SHAP_Rank"],
        shap_comparison_df["Conditional_SHAP_Rank"],
    ).statistic
)
standard_order = shap_comparison_df.sort_values(
    "Standard_SHAP_Rank"
)["Feature"].tolist()
conditional_order = shap_comparison_df.sort_values(
    "Conditional_SHAP_Rank"
)["Feature"].tolist()
standard_top_set = set(standard_order[:XAI_COMPARISON_TOP_K])
conditional_top_set = set(conditional_order[:XAI_COMPARISON_TOP_K])
top_overlap_count = len(standard_top_set & conditional_top_set)
top_overlap_fraction = top_overlap_count / XAI_COMPARISON_TOP_K
exact_full_order_agreement = standard_order == conditional_order

shap_stability_metrics_df = pd.DataFrame(
    [
        {
            "Feature_Count": len(shap_comparison_df),
            "Top_K": XAI_COMPARISON_TOP_K,
            "Exact_Full_Order_Agreement": exact_full_order_agreement,
            "Spearman_Rank_Correlation": spearman_rank_correlation,
            "Kendall_Rank_Correlation": kendall_rank_correlation,
            "Top_Feature_Set_Overlap_Count": top_overlap_count,
            "Top_Feature_Set_Overlap": top_overlap_fraction,
            "Maximum_Absolute_Normalized_Importance_Change": float(
                shap_comparison_df[
                    "Normalized_Importance_Change"
                ].abs().max()
            ),
        }
    ]
)
shap_comparison_df.to_csv(
    XAI_TABLES_DIR / "standard_vs_conditional_shap_comparison.csv",
    index=False,
)
shap_stability_metrics_df.to_csv(
    XAI_TABLES_DIR / "standard_vs_conditional_shap_stability_metrics.csv",
    index=False,
)

comparison_plot_df = shap_comparison_df.sort_values(
    "Standard_SHAP_Rank", ascending=False
).reset_index(drop=True)
fig, ax = plt.subplots(
    figsize=(9, max(4, 0.7 * len(comparison_plot_df)))
)
for position, row in comparison_plot_df.iterrows():
    ax.plot(
        [row["Standard_SHAP_Rank"], row["Conditional_SHAP_Rank"]],
        [position, position],
        color="grey",
        linewidth=1.5,
    )
    ax.scatter(
        row["Standard_SHAP_Rank"],
        position,
        color="#2878B5",
        s=70,
        label="Standard TreeSHAP" if position == 0 else None,
    )
    ax.scatter(
        row["Conditional_SHAP_Rank"],
        position,
        color="#D95319",
        s=70,
        label="Conditional SHAP (CTREE)" if position == 0 else None,
    )
ax.set_yticks(range(len(comparison_plot_df)))
ax.set_yticklabels(comparison_plot_df["Feature"])
ax.set_xlabel("Feature rank (1 = most important)")
ax.set_title("Standard vs Conditional SHAP Rankings\nCatBoost raw margin")
ax.invert_xaxis()
ax.legend()
fig.tight_layout()
shap_comparison_plot_path = (
    COMPARISON_DIR / "standard_vs_conditional_shap_rank_comparison.png"
)
fig.savefig(shap_comparison_plot_path, dpi=250, bbox_inches="tight")
plt.close(fig)

print("Standard vs Conditional SHAP comparison")
display(shap_comparison_df)
print("Explanation-stability metrics")
display(shap_stability_metrics_df)
print(f"Saved: {shap_comparison_plot_path}")


## 5.4 EBM Analysis

Use an Explainable Boosting Machine as a complementary interpretable model for feature-shape and exploratory threshold analysis. EBM explains its own fitted response surface; agreement with SHAP is cross-model triangulation rather than validation of the final predictive model.


### 5.4.1 Prepare the EBM Dataset

Create a leakage-safe, unscaled EBM dataset from the selected original feature groups. Expand each selected categorical source feature to all corresponding encoded terms and preserve mappings back to the clinical source feature.


In [ ]:
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight


def encoded_columns_for_source(source_feature, encoded_columns):
    if source_feature in encoded_columns:
        return [source_feature]
    prefix = f"{source_feature}_"
    return [
        column for column in encoded_columns if column.startswith(prefix)
    ]


def expand_original_groups_to_encoded(source_features, encoded_columns):
    expanded = []
    for source_feature in source_features:
        for column in encoded_columns_for_source(
            source_feature, encoded_columns
        ):
            if column not in expanded:
                expanded.append(column)
    return expanded


def encoded_to_original_feature(encoded_feature):
    if encoded_feature in numeric_model_columns:
        return encoded_feature
    matches = [
        feature
        for feature in categorical_model_columns
        if encoded_feature.startswith(f"{feature}_")
    ]
    if not matches:
        raise KeyError(
            f"Could not map encoded feature '{encoded_feature}' to a source feature."
        )
    return max(matches, key=len)


ebm_source_features = list(selected_final_features)
ebm_features = expand_original_groups_to_encoded(
    ebm_source_features,
    list(X_train_tree.columns),
)
if not ebm_features:
    raise RuntimeError("No EBM features matched the final selected source features.")

X_train_ebm = X_train_tree[ebm_features].copy()
X_test_ebm = X_test_tree[ebm_features].copy()
ebm_leakage_matches = sorted(
    {
        encoded_to_original_feature(feature)
        for feature in ebm_features
    }.intersection(LEAKAGE_COLUMNS)
)
if ebm_leakage_matches:
    raise RuntimeError(
        f"Leakage features detected in EBM inputs: {ebm_leakage_matches}"
    )

ebm_input_features_df = pd.DataFrame(
    {
        "Encoded_Feature": ebm_features,
        "Source_Feature": [
            encoded_to_original_feature(feature) for feature in ebm_features
        ],
    }
)
ebm_input_features_df.to_csv(
    XAI_TABLES_DIR / "ebm_input_features.csv", index=False
)
print("EBM input features")
display(ebm_input_features_df)


### 5.4.2 Train EBM and Perform Sanity Evaluation

Fit EBM on training data, using training-derived class balancing when required. Keep interactions disabled when every reported curve and threshold must correspond to one feature. Treat held-out metrics as a sanity evaluation, not a replacement benchmark.


In [ ]:
ebm_sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
ebm_model = ExplainableBoostingClassifier(
    interactions=0,
    max_bins=64,
    outer_bags=8,
    learning_rate=0.03,
    max_rounds=3000,
    early_stopping_rounds=100,
    min_samples_leaf=4,
    n_jobs=1,
    random_state=RANDOM_STATE,
)
ebm_model.fit(
    X_train_ebm,
    y_train,
    sample_weight=ebm_sample_weights,
)
ebm_test_scores = np.asarray(
    ebm_model.predict_proba(X_test_ebm), dtype=float
)[:, 1]
ebm_test_predictions = (ebm_test_scores >= 0.5).astype(int)
ebm_metrics = binary_metrics(y_test, ebm_test_scores, threshold=0.5)
ebm_metrics_df = pd.DataFrame(
    [{"Model": "Explainable Boosting Machine", **ebm_metrics}]
)
ebm_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, ebm_test_predictions, labels=[0, 1]),
    index=["Actual_0", "Actual_1"],
    columns=["Predicted_0", "Predicted_1"],
)
ebm_metrics_df.to_csv(XAI_TABLES_DIR / "ebm_sanity_metrics.csv", index=False)
ebm_confusion_matrix_df.to_csv(
    XAI_TABLES_DIR / "ebm_sanity_confusion_matrix.csv"
)
print("EBM held-out sanity evaluation")
display(ebm_metrics_df)
display(ebm_confusion_matrix_df)


### 5.4.3 Global EBM Explanation and Feature Curves

Save global EBM term importance and feature curves. Convert transformed numeric axes back to clinical units using training-fitted inverse parameters where available.


In [ ]:
ebm_global_explanation = ebm_model.explain_global(
    name="EEG Abnormality EBM Global Explanation"
)
ebm_term_importances = np.asarray(ebm_model.term_importances(), dtype=float)
ebm_feature_importance_df = (
    pd.DataFrame(
        {
            "Feature": ebm_model.term_names_,
            "Source_Feature": [
                encoded_to_original_feature(feature)
                for feature in ebm_model.term_names_
            ],
            "Importance": ebm_term_importances,
        }
    )
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)
ebm_feature_importance_df["EBM_Importance_Rank"] = np.arange(
    1, len(ebm_feature_importance_df) + 1
)
ebm_feature_importance_df.to_csv(
    XAI_TABLES_DIR / "ebm_feature_importance.csv", index=False
)

importance_plot_df = ebm_feature_importance_df.sort_values(
    "Importance", ascending=True
)
fig, ax = plt.subplots(
    figsize=(9, max(4, 0.55 * len(importance_plot_df)))
)
ax.barh(
    importance_plot_df["Feature"],
    importance_plot_df["Importance"],
    color="#2878B5",
)
ax.set_xlabel("Mean absolute EBM contribution")
ax.set_title("EBM Global Feature Importance")
fig.tight_layout()
ebm_global_importance_path = EBM_OUTPUT_DIR / "ebm_global_importance.png"
fig.savefig(ebm_global_importance_path, dpi=250, bbox_inches="tight")
plt.close(fig)


def ebm_curve_data(feature):
    term_index = list(ebm_model.term_names_).index(feature)
    term_data = ebm_global_explanation.data(term_index)
    raw_names = np.asarray(term_data["names"])
    scores = np.asarray(term_data["scores"], dtype=float)
    numeric_names = pd.to_numeric(pd.Series(raw_names), errors="coerce").to_numpy(
        dtype=float
    )
    if np.isnan(numeric_names).any():
        raise RuntimeError(f"EBM curve names are not numeric for '{feature}'.")

    if len(numeric_names) == len(scores) + 1:
        x_values = (numeric_names[:-1] + numeric_names[1:]) / 2
    else:
        x_values = numeric_names[: len(scores)]

    source_feature = encoded_to_original_feature(feature)
    if source_feature in numeric_model_columns and feature == source_feature:
        x_values = inverse_numeric_to_clinical(source_feature, x_values)
        axis_label = f"{source_feature} (clinical scale)"
    else:
        axis_label = feature
    return {
        "term_index": term_index,
        "x_values": np.asarray(x_values, dtype=float),
        "scores": scores,
        "axis_label": axis_label,
        "source_feature": source_feature,
    }


ebm_curve_cache = {}
ebm_curve_paths = []
for feature in ebm_model.term_names_:
    curve = ebm_curve_data(feature)
    ebm_curve_cache[feature] = curve
    finite = np.isfinite(curve["x_values"]) & np.isfinite(curve["scores"])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(
        curve["x_values"][finite],
        curve["scores"][finite],
        marker="o",
        color="#D95319",
    )
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_xlabel(curve["axis_label"])
    ax.set_ylabel("EBM contribution to abnormal-EEG log-odds")
    ax.set_title(f"EBM Feature Curve: {feature}")
    fig.tight_layout()
    output_path = EBM_OUTPUT_DIR / f"ebm_curve_{safe_feature_filename(feature)}.png"
    fig.savefig(output_path, dpi=250, bbox_inches="tight")
    plt.close(fig)
    ebm_curve_paths.append(output_path)

print("EBM global feature importance")
display(ebm_feature_importance_df)
print(f"Saved: {ebm_global_importance_path}")
print(f"Saved {len(ebm_curve_paths)} EBM feature curves.")


### 5.4.4 Extract Exploratory EBM Thresholds

Identify exploratory numeric transitions using a prespecified sustained terminal-sign rule: select the first zero crossing after which all later curve contributions retain the terminal direction. Use the largest adjacent change only as an explicitly labelled fallback. Report categorical level effects separately rather than presenting a dummy split as a clinical threshold. These model-derived transitions are not clinical cut-offs.


In [ ]:
def find_sustained_terminal_transition(values, effects):
    values = np.asarray(values, dtype=float)
    effects = np.asarray(effects, dtype=float)
    nonzero = np.flatnonzero(~np.isclose(effects, 0.0))
    if len(effects) < 2 or not len(nonzero):
        return None, 0

    terminal_sign = int(np.sign(effects[nonzero[-1]]))
    if terminal_sign > 0:
        candidates = [
            i for i in range(len(effects) - 1)
            if effects[i] <= 0 and effects[i + 1] > 0
            and np.all(effects[i + 1:] >= 0)
        ]
    else:
        candidates = [
            i for i in range(len(effects) - 1)
            if effects[i] >= 0 and effects[i + 1] < 0
            and np.all(effects[i + 1:] <= 0)
        ]
    return (int(candidates[0]), terminal_sign) if candidates else (None, terminal_sign)


def derive_ebm_numeric_threshold(feature, curve):
    x_values = np.asarray(curve["x_values"], dtype=float)
    scores = np.asarray(curve["scores"], dtype=float)
    finite = np.isfinite(x_values) & np.isfinite(scores)
    x_values = x_values[finite]
    scores = scores[finite]

    if len(scores) < 2 or np.allclose(scores, scores[0]):
        return {
            "Feature": feature,
            "EBM_Threshold": np.nan,
            "Threshold_Direction": "No clear single transition",
            "Threshold_Method": "Insufficient EBM curve variation",
            "Confidence_Level": "Weak / exploratory",
        }

    adjacent_changes = np.abs(np.diff(scores))
    transition_index, terminal_sign = find_sustained_terminal_transition(
        x_values, scores
    )
    crossing_indices = np.where(
        ((scores[:-1] <= 0) & (scores[1:] > 0))
        | ((scores[:-1] >= 0) & (scores[1:] < 0))
    )[0]
    if transition_index is not None:
        method = "First zero crossing into sustained terminal-sign region"
        confidence = "Exploratory / sustained"
    elif len(crossing_indices):
        transition_index = int(
            crossing_indices[np.argmax(adjacent_changes[crossing_indices])]
        )
        method = "Fallback: largest EBM zero-contribution crossing"
        confidence = "Weak / exploratory"
    else:
        transition_index = int(np.argmax(adjacent_changes))
        method = "Fallback: largest adjacent EBM score change"
        confidence = "Weak / exploratory"

    threshold = float(
        (x_values[transition_index] + x_values[transition_index + 1]) / 2
    )
    if terminal_sign > 0 and method.startswith("First zero crossing"):
        direction = "Above transition associated with higher predicted abnormal-EEG risk"
    elif terminal_sign < 0 and method.startswith("First zero crossing"):
        direction = "Below transition associated with higher predicted abnormal-EEG risk"
    else:
        lower_score = float(np.mean(scores[: transition_index + 1]))
        upper_score = float(np.mean(scores[transition_index + 1 :]))
        direction = (
            "Above transition associated with higher predicted abnormal-EEG risk"
            if upper_score > lower_score
            else "Below transition associated with higher predicted abnormal-EEG risk"
            if lower_score > upper_score
            else "No clear single transition"
        )
    return {
        "Feature": feature,
        "EBM_Threshold": threshold,
        "Threshold_Direction": direction,
        "Threshold_Method": method,
        "Confidence_Level": confidence,
    }


ebm_numeric_features = [
    feature
    for feature in ebm_model.term_names_
    if encoded_to_original_feature(feature) in numeric_model_columns
    and feature == encoded_to_original_feature(feature)
]
ebm_numeric_threshold_df = pd.DataFrame(
    [
        derive_ebm_numeric_threshold(feature, ebm_curve_cache[feature])
        for feature in ebm_numeric_features
    ]
)

ebm_categorical_effect_rows = []
for feature in ebm_model.term_names_:
    source_feature = encoded_to_original_feature(feature)
    if source_feature not in categorical_model_columns:
        continue
    curve = ebm_curve_cache[feature]
    x_values = np.asarray(curve["x_values"], dtype=float)
    scores = np.asarray(curve["scores"], dtype=float)
    finite = np.isfinite(x_values) & np.isfinite(scores)
    x_values = x_values[finite]
    scores = scores[finite]
    if len(scores) < 2:
        presence_effect = np.nan
    else:
        absence_score = float(scores[np.argmin(np.abs(x_values - 0.0))])
        presence_score = float(scores[np.argmin(np.abs(x_values - 1.0))])
        presence_effect = presence_score - absence_score
    level = feature[len(source_feature) + 1 :]
    ebm_categorical_effect_rows.append(
        {
            "Encoded_Feature": feature,
            "Source_Feature": source_feature,
            "Category_Level": level,
            "Presence_Minus_Absence_Effect": presence_effect,
            "Direction": (
                "Presence associated with higher EBM-predicted abnormal-EEG risk"
                if pd.notna(presence_effect) and presence_effect > 0
                else "Presence associated with lower EBM-predicted abnormal-EEG risk"
                if pd.notna(presence_effect) and presence_effect < 0
                else "No clear categorical effect"
            ),
        }
    )
ebm_categorical_effect_df = pd.DataFrame(ebm_categorical_effect_rows)

ebm_numeric_threshold_df.to_csv(
    XAI_TABLES_DIR / "ebm_exploratory_numeric_thresholds.csv", index=False
)
ebm_categorical_effect_df.to_csv(
    XAI_TABLES_DIR / "ebm_categorical_level_effects.csv", index=False
)
print("Exploratory EBM numeric transitions")
display(ebm_numeric_threshold_df)
print("EBM categorical level effects")
display(ebm_categorical_effect_df)


### 5.4.5 EBM vs SHAP Threshold Comparison

Compare EBM and Standard SHAP numeric transitions using the same sustained terminal-sign rule. Compare the resulting transition locations and directions as cross-model triangulation. Label all transitions as exploratory model-derived signals, not validated clinical cut-offs.


In [ ]:
def derive_standard_shap_numeric_threshold(feature):
    feature_index = selected_final_features.index(feature)
    feature_values = standard_shap_display_X[feature].to_numpy(dtype=float)
    shap_values = standard_shap_values[:, feature_index]
    dependence = (
        pd.DataFrame(
            {"Feature_Value": feature_values, "SHAP_Value": shap_values}
        )
        .groupby("Feature_Value", as_index=False)["SHAP_Value"]
        .mean()
        .sort_values("Feature_Value")
        .reset_index(drop=True)
    )
    if len(dependence) < 2:
        return {
            "Feature": feature,
            "SHAP_Dependence_Threshold": np.nan,
            "SHAP_Threshold_Direction": "No clear observed sign transition",
            "SHAP_Threshold_Method": "Insufficient distinct held-out values",
        }

    values = dependence["Feature_Value"].to_numpy(dtype=float)
    effects = dependence["SHAP_Value"].to_numpy(dtype=float)
    transition_index, terminal_sign = find_sustained_terminal_transition(
        values, effects
    )
    crossing_indices = np.where(
        ((effects[:-1] <= 0) & (effects[1:] > 0))
        | ((effects[:-1] >= 0) & (effects[1:] < 0))
    )[0]
    if transition_index is None:
        return {
            "Feature": feature,
            "SHAP_Dependence_Threshold": np.nan,
            "SHAP_Threshold_Direction": "No clear observed sign transition",
            "SHAP_Threshold_Method": (
                "Observed sign changes did not enter a sustained terminal-sign region"
                if len(crossing_indices)
                else "No held-out Standard SHAP sign change"
            ),
        }
    threshold = float(
        (values[transition_index] + values[transition_index + 1]) / 2
    )
    if terminal_sign > 0:
        direction = "Above transition associated with higher predicted abnormal-EEG risk"
    else:
        direction = "Below transition associated with higher predicted abnormal-EEG risk"
    return {
        "Feature": feature,
        "SHAP_Dependence_Threshold": threshold,
        "SHAP_Threshold_Direction": direction,
        "SHAP_Threshold_Method": "First observed sign-change midpoint into sustained terminal-sign region",
    }


standard_shap_numeric_threshold_df = pd.DataFrame(
    [
        derive_standard_shap_numeric_threshold(feature)
        for feature in ebm_numeric_features
    ]
)
standard_shap_numeric_threshold_df.to_csv(
    XAI_TABLES_DIR / "standard_shap_exploratory_numeric_thresholds.csv",
    index=False,
)

ebm_vs_shap_threshold_df = ebm_numeric_threshold_df.merge(
    standard_shap_numeric_threshold_df,
    on="Feature",
    how="outer",
    validate="one_to_one",
)


def compare_ebm_shap_threshold(row):
    ebm_threshold = row["EBM_Threshold"]
    shap_threshold = row["SHAP_Dependence_Threshold"]
    ebm_direction = row["Threshold_Direction"]
    shap_direction = row["SHAP_Threshold_Direction"]
    if pd.isna(ebm_threshold) or pd.isna(shap_threshold):
        return pd.Series(
            {
                "Direction_Agreement": False,
                "Absolute_Threshold_Difference": np.nan,
                "Relative_Threshold_Difference": np.nan,
                "Agreement": "Not assessable",
            }
        )
    direction_agreement = ebm_direction == shap_direction
    absolute_difference = abs(float(ebm_threshold) - float(shap_threshold))
    relative_difference = absolute_difference / max(abs(float(ebm_threshold)), 1e-8)
    if direction_agreement and relative_difference <= 0.10:
        agreement = "Agree"
    elif direction_agreement and relative_difference <= 0.20:
        agreement = "Partially agree"
    else:
        agreement = "Disagree"
    return pd.Series(
        {
            "Direction_Agreement": direction_agreement,
            "Absolute_Threshold_Difference": absolute_difference,
            "Relative_Threshold_Difference": relative_difference,
            "Agreement": agreement,
        }
    )


ebm_shap_agreement = ebm_vs_shap_threshold_df.apply(
    compare_ebm_shap_threshold, axis=1
)
ebm_vs_shap_threshold_df = pd.concat(
    [ebm_vs_shap_threshold_df, ebm_shap_agreement], axis=1
)
ebm_vs_shap_threshold_df.to_csv(
    XAI_TABLES_DIR / "ebm_vs_shap_exploratory_threshold_comparison.csv",
    index=False,
)
print("EBM vs Standard SHAP exploratory numeric-transition comparison")
display(ebm_vs_shap_threshold_df)


## 5.5 DML Analysis

Use DML only as an exploratory adjusted-association layer after defining a defensible estimand, temporal ordering, treatment representation, and observed confounder set. DML is not a prediction benchmark and cannot establish causality in a retrospective observational dataset with possible unmeasured confounding.


### 5.5.1 Prepare DML Inputs

Define candidate treatments and confounders without leakage. Use training data only for exploratory DML estimation, preserve clinically interpretable units where appropriate, and document categorical reference groups or grouped treatments.


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests


def encode_dml_feature_frame(X_frame):
    """Encode selected model inputs without using held-out test information."""
    encoded_parts = []
    source_columns = {}
    reference_levels = {}

    for feature in selected_final_features:
        if feature in selected_final_categorical_features:
            levels = list(xai_category_levels[feature])
            reference_levels[feature] = levels[0]
            categorical = pd.Categorical(
                X_frame[feature].astype(str), categories=levels, ordered=False
            )
            if np.any(categorical.codes < 0):
                raise ValueError(
                    f"DML feature '{feature}' contains a category absent from "
                    "the training-derived level map."
                )
            columns = []
            for level in levels[1:]:
                column_name = f"{feature} == {level}"
                encoded_parts.append(
                    pd.Series(
                        (X_frame[feature].astype(str) == level).astype(float),
                        index=X_frame.index,
                        name=column_name,
                    )
                )
                columns.append(column_name)
            source_columns[feature] = columns
        else:
            column_name = feature
            clinical_values = inverse_numeric_to_clinical(
                feature,
                pd.to_numeric(X_frame[feature], errors="raise").to_numpy(float),
            )
            encoded_parts.append(
                pd.Series(clinical_values, index=X_frame.index, name=column_name)
            )
            source_columns[feature] = [column_name]

    encoded = pd.concat(encoded_parts, axis=1).astype(float)
    if not np.isfinite(encoded.to_numpy()).all():
        raise RuntimeError("DML design contains non-finite values.")
    return encoded, source_columns, reference_levels


dml_design_X, dml_source_columns, dml_reference_levels = encode_dml_feature_frame(
    X_train_final
)
dml_outcome_y = pd.Series(
    np.asarray(y_train, dtype=int), index=X_train_final.index, name="EEG_ABNORMAL"
)

dml_candidate_rows = []
for source_feature, encoded_columns in dml_source_columns.items():
    if source_feature in selected_final_categorical_features:
        reference_level = dml_reference_levels[source_feature]
        for encoded_column in encoded_columns:
            level = encoded_column.split(" == ", 1)[1]
            dml_candidate_rows.append(
                {
                    "Candidate": encoded_column,
                    "Source_Feature": source_feature,
                    "Treatment_Column": encoded_column,
                    "Treatment_Type": "binary",
                    "Contrast": f"{level} versus {reference_level}",
                    "Unit": "indicator contrast",
                    "Reference_Level": reference_level,
                }
            )
    else:
        dml_candidate_rows.append(
            {
                "Candidate": source_feature,
                "Source_Feature": source_feature,
                "Treatment_Column": source_feature,
                "Treatment_Type": "continuous",
                "Contrast": "one training-standard-deviation increase",
                "Unit": "training SD",
                "Reference_Level": None,
            }
        )

dml_candidate_df = pd.DataFrame(dml_candidate_rows)
dml_candidate_df["Confounder_Columns"] = dml_candidate_df["Source_Feature"].map(
    lambda source: json.dumps(
        [
            column
            for other_source, columns in dml_source_columns.items()
            if other_source != source
            for column in columns
        ]
    )
)
dml_candidate_df.to_csv(
    DML_OUTPUT_DIR / "dml_candidate_definitions.csv", index=False
)
print("Exploratory DML candidate definitions (training data only)")
display(dml_candidate_df)


### 5.5.2 Run Cross-Fitted Residualization DML

Estimate exploratory adjusted effects with cross-fitted nuisance models and a clearly documented final-stage estimator. Report effect units, confidence intervals, p-values, overlap diagnostics, and nuisance-model diagnostics where feasible.


In [ ]:
def make_dml_catboost_classifier(seed):
    return CatBoostClassifier(
        iterations=250,
        depth=3,
        learning_rate=0.04,
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=1,
    )


def make_dml_catboost_regressor(seed):
    return CatBoostRegressor(
        iterations=250,
        depth=3,
        learning_rate=0.04,
        loss_function="RMSE",
        random_seed=seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=1,
    )


def run_cross_fitted_dml(candidate_row):
    source_feature = candidate_row["Source_Feature"]
    treatment_column = candidate_row["Treatment_Column"]
    treatment_type = candidate_row["Treatment_Type"]
    confounder_columns = [
        column
        for other_source, columns in dml_source_columns.items()
        if other_source != source_feature
        for column in columns
    ]
    if not confounder_columns:
        raise RuntimeError(
            f"DML candidate '{candidate_row['Candidate']}' has no remaining "
            "confounders after excluding its source feature."
        )

    W = dml_design_X[confounder_columns]
    treatment_raw = dml_design_X[treatment_column].to_numpy(float)
    treatment_sd = float(np.std(treatment_raw, ddof=1))
    if treatment_sd <= 0 or not np.isfinite(treatment_sd):
        raise RuntimeError(
            f"DML candidate '{candidate_row['Candidate']}' has no usable "
            "training variation."
        )
    if treatment_type == "continuous":
        treatment = (treatment_raw - treatment_raw.mean()) / treatment_sd
    else:
        treatment = treatment_raw
        if set(np.unique(treatment)) != {0.0, 1.0}:
            raise RuntimeError(
                f"Binary DML candidate '{candidate_row['Candidate']}' does "
                "not contain both indicator levels."
            )

    outcome = dml_outcome_y.to_numpy(int)
    outcome_hat = np.full(len(W), np.nan, dtype=float)
    treatment_hat = np.full(len(W), np.nan, dtype=float)
    splitter = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=RANDOM_STATE
    )

    for fold, (train_index, validation_index) in enumerate(
        splitter.split(W, outcome), start=1
    ):
        outcome_model = make_dml_catboost_classifier(RANDOM_STATE + fold)
        outcome_model.fit(W.iloc[train_index], outcome[train_index])
        outcome_hat[validation_index] = outcome_model.predict_proba(
            W.iloc[validation_index]
        )[:, 1]

        if treatment_type == "binary":
            if len(np.unique(treatment[train_index])) < 2:
                raise RuntimeError(
                    f"A DML training fold for '{candidate_row['Candidate']}' "
                    "contains only one treatment level."
                )
            treatment_model = make_dml_catboost_classifier(
                RANDOM_STATE + 100 + fold
            )
            treatment_model.fit(W.iloc[train_index], treatment[train_index])
            treatment_hat[validation_index] = treatment_model.predict_proba(
                W.iloc[validation_index]
            )[:, 1]
        else:
            treatment_model = make_dml_catboost_regressor(
                RANDOM_STATE + 100 + fold
            )
            treatment_model.fit(W.iloc[train_index], treatment[train_index])
            treatment_hat[validation_index] = treatment_model.predict(
                W.iloc[validation_index]
            )

    if not (
        np.isfinite(outcome_hat).all() and np.isfinite(treatment_hat).all()
    ):
        raise RuntimeError(
            f"Cross-fitting left non-finite nuisance predictions for "
            f"'{candidate_row['Candidate']}'."
        )

    outcome_residual = outcome - outcome_hat
    treatment_residual = treatment - treatment_hat
    if np.std(treatment_residual, ddof=1) <= 1e-10:
        raise RuntimeError(
            f"Residualized treatment variation collapsed for "
            f"'{candidate_row['Candidate']}'."
        )

    final_stage = sm.OLS(
        outcome_residual, treatment_residual.reshape(-1, 1)
    ).fit(cov_type="HC3")
    confidence_interval = final_stage.conf_int(alpha=0.05)[0]

    result = {
        "Candidate": candidate_row["Candidate"],
        "Source_Feature": source_feature,
        "Treatment_Type": treatment_type,
        "Contrast": candidate_row["Contrast"],
        "Reference_Level": candidate_row["Reference_Level"],
        "DML_Effect": float(final_stage.params[0]),
        "CI_95_Lower": float(confidence_interval[0]),
        "CI_95_Upper": float(confidence_interval[1]),
        "Robust_SE_HC3": float(final_stage.bse[0]),
        "P_Value": float(final_stage.pvalues[0]),
        "Training_N": len(W),
        "Confounder_Count": len(confounder_columns),
        "Treatment_Training_SD_Raw": treatment_sd,
        "Outcome_Nuisance_AUC": float(roc_auc_score(outcome, outcome_hat)),
    }
    if treatment_type == "binary":
        result.update(
            {
                "Treatment_Prevalence": float(treatment.mean()),
                "Treatment_Nuisance_AUC": float(
                    roc_auc_score(treatment, treatment_hat)
                ),
                "Treatment_Nuisance_RMSE": np.nan,
                "Treatment_Nuisance_R2": np.nan,
                "Propensity_Min": float(treatment_hat.min()),
                "Propensity_Max": float(treatment_hat.max()),
                "Propensity_Within_0.05_0.95": float(
                    np.mean((treatment_hat >= 0.05) & (treatment_hat <= 0.95))
                ),
            }
        )
    else:
        result.update(
            {
                "Treatment_Prevalence": np.nan,
                "Treatment_Nuisance_AUC": np.nan,
                "Treatment_Nuisance_RMSE": float(
                    np.sqrt(mean_squared_error(treatment, treatment_hat))
                ),
                "Treatment_Nuisance_R2": float(
                    r2_score(treatment, treatment_hat)
                ),
                "Propensity_Min": np.nan,
                "Propensity_Max": np.nan,
                "Propensity_Within_0.05_0.95": np.nan,
            }
        )
    return result


dml_results = [
    run_cross_fitted_dml(row)
    for _, row in dml_candidate_df.iterrows()
]
dml_results_df = pd.DataFrame(dml_results)
rejected, adjusted_p_values, _, _ = multipletests(
    dml_results_df["P_Value"].to_numpy(float),
    alpha=0.05,
    method="fdr_bh",
)
dml_results_df["FDR_BH_Adjusted_P_Value"] = adjusted_p_values
dml_results_df["FDR_BH_Significant_0.05"] = rejected
dml_results_df = dml_results_df.sort_values(
    ["FDR_BH_Adjusted_P_Value", "P_Value", "Candidate"]
).reset_index(drop=True)


### 5.5.3 Save and Summarize DML Results

Save the exploratory effect table and uncertainty plot. Separate positive, negative, and unclear adjusted signals, address multiple testing where applicable, and state the limitations of post-selection, small samples, observed-confounder adjustment, and unmeasured confounding.


In [ ]:
dml_results_path = DML_OUTPUT_DIR / "exploratory_dml_results.csv"
dml_results_df.to_csv(dml_results_path, index=False)

fig, ax = plt.subplots(
    figsize=(8.5, max(4.5, 0.65 * len(dml_results_df) + 1.8))
)
dml_plot_df = dml_results_df.iloc[::-1].reset_index(drop=True)
y_positions = np.arange(len(dml_plot_df))
effects = dml_plot_df["DML_Effect"].to_numpy(float)
lower_errors = effects - dml_plot_df["CI_95_Lower"].to_numpy(float)
upper_errors = dml_plot_df["CI_95_Upper"].to_numpy(float) - effects
ax.errorbar(
    effects,
    y_positions,
    xerr=np.vstack([lower_errors, upper_errors]),
    fmt="o",
    color="#2F6B8A",
    ecolor="#6A96AD",
    capsize=4,
)
ax.axvline(0, color="grey", linestyle="--", linewidth=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(dml_plot_df["Candidate"])
ax.set_xlabel("Exploratory residualized effect on abnormal-EEG probability")
ax.set_title("Cross-Fitted Residualization DML (Training Data Only)")
fig.tight_layout()
dml_forest_path = DML_OUTPUT_DIR / "exploratory_dml_uncertainty.png"
fig.savefig(dml_forest_path, dpi=250, bbox_inches="tight")
plt.close(fig)

dml_limitations = [
    "This exploratory DML analysis uses training data only and does not alter the frozen predictive model.",
    "Residualized associations are not causal estimates unless untestable identification assumptions hold.",
    "The selected predictive features are not a prespecified causal adjustment set; unmeasured and temporal confounding remain likely.",
    "Categorical effects compare each observed non-reference training level with the training-derived reference level.",
    "Continuous effects are reported per one training-standard-deviation increase on the clinical display scale.",
    "The small, single-centre retrospective cohort limits nuisance-model stability, overlap assessment, and generalizability.",
    "Benjamini-Hochberg adjustment is reported across the exploratory candidate tests; results require independent confirmation.",
]
pd.DataFrame({"Limitation": dml_limitations}).to_csv(
    DML_OUTPUT_DIR / "dml_limitations.csv", index=False
)

print("Exploratory DML results (not causal claims)")
display(dml_results_df)
print(f"Saved DML results: {dml_results_path}")
print(f"Saved DML uncertainty plot: {dml_forest_path}")
for limitation in dml_limitations:
    print(f"- {limitation}")


# Compatibility helper retained for the protected Sodium Analysis section.
# It is defined here but the protected section is neither edited nor executed.
def build_shap_explanation(model_key, model, X_background, X_explain):
    if model_key != "catboost":
        raise ValueError(
            "The revised Section 5 compatibility helper supports the frozen "
            "CatBoost pathway used by this EEG project."
        )
    feature_names = list(X_explain.columns)
    categorical_features = [
        feature
        for feature in feature_names
        if feature in categorical_model_columns
        or not pd.api.types.is_numeric_dtype(X_explain[feature])
    ]
    category_levels = training_category_levels(
        X_background, categorical_features
    )
    display_frame, category_label_lists = xai_display_frame(
        X_explain, categorical_features, category_levels
    )
    pool = Pool(X_explain, cat_features=categorical_features)
    raw_shap = model.get_feature_importance(pool, type="ShapValues")
    explanation = shap.Explanation(
        values=np.asarray(raw_shap[:, :-1], dtype=float),
        base_values=np.asarray(raw_shap[:, -1], dtype=float),
        data=display_frame.to_numpy(float),
        feature_names=feature_names,
    )
    category_maps = {
        feature: dict(enumerate(levels))
        for feature, levels in category_label_lists.items()
    }
    return explanation, category_maps


xai_outputs = {
    "standard_shap": str(STANDARD_SHAP_DIR),
    "conditional_shap": str(CONDITIONAL_SHAP_DIR),
    "shap_comparison": str(COMPARISON_DIR),
    "ebm": str(EBM_OUTPUT_DIR),
    "dml": str(DML_OUTPUT_DIR),
}
pd.Series(xai_outputs, name="Output_Directory").to_csv(
    XAI_OUTPUT_DIR / "xai_output_manifest.csv"
)


# 6. Sodium Analysis

This exploratory section analyzes `SODIUM` separately because prior CatBoost feature selection selected it and current BorutaSHAP marked it as tentative. The main final CatBoost model remains unchanged and continues to use BorutaSHAP accepted features only.

### 6.1 Sodium Distribution by EEG Abnormality

This subsection describes the univariate sodium distribution by `EEG_ABNORMAL`, including median, IQR, mean, standard deviation, missing count, and class-stratified visualizations.

In [ ]:
SODIUM_OUTPUT_DIR = OUTPUT_DIR / "sodium_analysis"
SODIUM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "SODIUM" not in model_data.columns:
    raise KeyError("SODIUM is not available in the retained modelling dataset.")

sodium_raw = model_data[[TARGET_BINARY, "SODIUM"]].copy()
sodium_raw["SODIUM"] = coerce_numeric(sodium_raw["SODIUM"])
sodium_raw["EEG class"] = np.where(sodium_raw[TARGET_BINARY].astype(int) == 1, "Abnormal EEG", "Normal EEG")

sodium_distribution = (
    sodium_raw.groupby("EEG class", observed=False)["SODIUM"]
    .agg(
        n="size",
        observed="count",
        missing_count=lambda s: int(s.isna().sum()),
        median="median",
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
        mean="mean",
        std="std",
    )
    .reset_index()
)
sodium_distribution["IQR"] = sodium_distribution["q3"] - sodium_distribution["q1"]
sodium_distribution = sodium_distribution[
    ["EEG class", "n", "observed", "missing_count", "median", "q1", "q3", "IQR", "mean", "std"]
].round(3)

print("Sodium distribution by EEG_ABNORMAL class")
display(sodium_distribution)
sodium_distribution.to_csv(SODIUM_OUTPUT_DIR / "sodium_distribution_by_eeg_abnormal.csv", index=False)

plot_df = sodium_raw.dropna(subset=["SODIUM"]).copy()

fig, ax = plt.subplots(figsize=(7, 5))
sns.violinplot(data=plot_df, x="EEG class", y="SODIUM", inner="box", cut=0, ax=ax)
ax.axhspan(135, 145, color="tab:green", alpha=0.08, label="Normal sodium range 135-145")
ax.set_title("Sodium distribution by EEG abnormality class")
ax.set_xlabel("")
ax.set_ylabel("Sodium (mmol/L)")
ax.legend(loc="best")
plt.tight_layout()
sodium_violin_path = SODIUM_OUTPUT_DIR / "sodium_violin_by_eeg_abnormal.png"
plt.savefig(sodium_violin_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=plot_df, x="SODIUM", hue="EEG class", bins=20, kde=True, stat="density", common_norm=False, alpha=0.35, ax=ax)
ax.axvline(135, color="tab:orange", linestyle="--", linewidth=1.5, label="135 mmol/L")
ax.axvline(145, color="tab:red", linestyle="--", linewidth=1.5, label="145 mmol/L")
ax.set_title("Sodium histogram by EEG abnormality class")
ax.set_xlabel("Sodium (mmol/L)")
plt.tight_layout()
sodium_hist_path = SODIUM_OUTPUT_DIR / "sodium_histogram_by_eeg_abnormal.png"
plt.savefig(sodium_hist_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Saved sodium distribution plots to: {SODIUM_OUTPUT_DIR}")

### 6.2 Clinical Sodium Ranges

Sodium is grouped using common clinical cutoffs: hyponatremia `<135`, normal `135-145`, and hypernatremia `>145`. These summaries are descriptive and do not change the final model.

In [ ]:
def sodium_range_group(value):
    if pd.isna(value):
        return "Missing"
    if value < 135:
        return "Hyponatremia (<135)"
    if value <= 145:
        return "Normal (135-145)"
    return "Hypernatremia (>145)"

sodium_range_order = ["Hyponatremia (<135)", "Normal (135-145)", "Hypernatremia (>145)", "Missing"]
sodium_ranges = sodium_raw.copy()
sodium_ranges["sodium_group"] = pd.Categorical(
    sodium_ranges["SODIUM"].apply(sodium_range_group),
    categories=sodium_range_order,
    ordered=True,
)

sodium_range_summary = (
    sodium_ranges.groupby("sodium_group", observed=False)
    .agg(
        n=(TARGET_BINARY, "size"),
        abnormal_eeg_count=(TARGET_BINARY, "sum"),
        abnormal_eeg_rate=(TARGET_BINARY, "mean"),
    )
    .reset_index()
)
sodium_range_summary["abnormal_eeg_count"] = sodium_range_summary["abnormal_eeg_count"].astype(int)
sodium_range_summary["abnormal_eeg_rate_pct"] = (sodium_range_summary["abnormal_eeg_rate"] * 100).round(2)

print("Abnormal EEG rate by clinical sodium range")
display(sodium_range_summary[["sodium_group", "n", "abnormal_eeg_count", "abnormal_eeg_rate_pct"]])
sodium_range_summary.to_csv(SODIUM_OUTPUT_DIR / "sodium_clinical_range_abnormal_rate.csv", index=False)


### 6.3 Sodium Model Signal

This subsection fits a sensitivity CatBoost model using BorutaSHAP accepted features plus `SODIUM`. It is used only to inspect sodium signal and SHAP dependence; it does not replace or alter the main final CatBoost model.

In [ ]:
if "SODIUM" not in X_train_catboost.columns:
    raise KeyError("SODIUM is not present after preprocessing; sensitivity model cannot be fitted.")

sodium_sensitivity_features = list(dict.fromkeys(borutashap_accepted_features + ["SODIUM"]))
main_accepted_features_for_sodium_section = borutashap_accepted_features.copy()

X_train_sodium_sensitivity = X_train_catboost[sodium_sensitivity_features].copy()
X_test_sodium_sensitivity = X_test_catboost[sodium_sensitivity_features].copy()

sodium_sensitivity_model = make_model("catboost", params=best_tuning_params)
fit_model(sodium_sensitivity_model, "catboost", X_train_sodium_sensitivity, y_train)

sodium_shap_explanation, sodium_shap_category_maps = build_shap_explanation(
    "catboost",
    sodium_sensitivity_model,
    X_train_sodium_sensitivity,
    X_test_sodium_sensitivity,
)
sodium_shap_display_df = pd.DataFrame(
    sodium_shap_explanation.data,
    columns=list(X_test_sodium_sensitivity.columns),
    index=X_test_sodium_sensitivity.index,
)

sodium_feature_index = list(X_test_sodium_sensitivity.columns).index("SODIUM")
sodium_shap_values = sodium_shap_explanation.values[:, sodium_feature_index]
sodium_shap_signal_summary = pd.DataFrame(
    [
        {
            "feature": "SODIUM",
            "mean_abs_shap": float(np.abs(sodium_shap_values).mean()),
            "mean_shap": float(np.mean(sodium_shap_values)),
            "min_shap": float(np.min(sodium_shap_values)),
            "max_shap": float(np.max(sodium_shap_values)),
        }
    ]
)

plt.figure(figsize=(7, 5))
shap.dependence_plot(
    "SODIUM",
    sodium_shap_explanation.values,
    sodium_shap_display_df,
    feature_names=list(X_test_sodium_sensitivity.columns),
    show=False,
    interaction_index=None,
)
plt.title("SODIUM SHAP dependence - sensitivity CatBoost model")
plt.tight_layout()
sodium_dependence_path = SODIUM_OUTPUT_DIR / "sodium_shap_dependence_sensitivity_catboost.png"
plt.savefig(sodium_dependence_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close()

sodium_bin_edges = [-np.inf, 130, 135, 140, 145, np.inf]
sodium_bin_labels = ["<130", "130-134", "135-140", "141-145", ">145"]
sodium_binned = sodium_raw.dropna(subset=["SODIUM"]).copy()
sodium_binned["sodium_bin"] = pd.cut(sodium_binned["SODIUM"], bins=sodium_bin_edges, labels=sodium_bin_labels)
sodium_bin_summary = (
    sodium_binned.groupby("sodium_bin", observed=False)
    .agg(n=(TARGET_BINARY, "size"), abnormal_eeg_count=(TARGET_BINARY, "sum"), abnormal_eeg_rate=(TARGET_BINARY, "mean"))
    .reset_index()
)
sodium_bin_summary["abnormal_eeg_count"] = sodium_bin_summary["abnormal_eeg_count"].astype(int)
sodium_bin_summary["abnormal_eeg_rate_pct"] = (sodium_bin_summary["abnormal_eeg_rate"] * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=sodium_bin_summary, x="sodium_bin", y="abnormal_eeg_rate_pct", color="steelblue", ax=ax)
ax.set_title("Abnormal EEG rate across sodium bins")
ax.set_xlabel("Sodium bin (mmol/L)")
ax.set_ylabel("Abnormal EEG rate (%)")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", fontsize=9)
plt.tight_layout()
sodium_bin_rate_path = SODIUM_OUTPUT_DIR / "sodium_binned_abnormal_eeg_rate.png"
plt.savefig(sodium_bin_rate_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Sodium sensitivity model features:")
print(sodium_sensitivity_features)
print("Sodium SHAP signal summary:")
display(sodium_shap_signal_summary.round(4))
print("Binned abnormal EEG rate across sodium ranges:")
display(sodium_bin_summary)

sodium_shap_signal_summary.to_csv(SODIUM_OUTPUT_DIR / "sodium_shap_signal_summary.csv", index=False)
sodium_bin_summary.to_csv(SODIUM_OUTPUT_DIR / "sodium_binned_abnormal_rate.csv", index=False)
print(f"Saved sodium SHAP dependence plot: {sodium_dependence_path}")
print(f"Saved sodium binned rate plot: {sodium_bin_rate_path}")


### 6.4 Sensitivity Model Comparison

The main model and sodium sensitivity model are compared using the same train/test split. Thresholds are selected from out-of-fold training predictions only; the held-out test set is used only for final comparison.

In [ ]:
def oof_scores_for_catboost_feature_set(features, params=None):
    X_matrix = X_train_catboost[features].copy()
    y_values = pd.Series(y_train).astype(int)
    scores = pd.Series(index=y_values.index, dtype=float)
    candidate_categorical = [col for col in catboost_categorical_features if col in features]
    for train_idx, valid_idx in cv.split(X_matrix, y_values):
        model = make_model("catboost", params=params)
        model.fit(
            X_matrix.iloc[train_idx].copy(),
            y_values.iloc[train_idx].copy(),
            cat_features=[col for col in candidate_categorical if col in X_matrix.columns],
            verbose=False,
        )
        scores.iloc[valid_idx] = predict_scores(model, X_matrix.iloc[valid_idx].copy())
    return scores.astype(float)


def fit_catboost_feature_set(features, params):
    X_matrix = X_train_catboost[features].copy()
    model = make_model("catboost", params=params)
    model.fit(
        X_matrix,
        y_train,
        cat_features=[col for col in catboost_categorical_features if col in features],
        verbose=False,
    )
    return model


def evaluate_catboost_sensitivity_feature_set(model_label, features):
    tuning_rows = []
    tuning_oof_by_params = []
    for params in ParameterGrid(TUNING_GRIDS["catboost"]):
        train_oof_scores = oof_scores_for_catboost_feature_set(features, params=params)
        row = {"params": params}
        row.update(binary_metrics(y_train, train_oof_scores, threshold=0.5))
        tuning_rows.append(row)
        tuning_oof_by_params.append((params, train_oof_scores))

    tuning_results = pd.DataFrame(tuning_rows).sort_values(metric_priority, ascending=False).reset_index(drop=True)
    best_params = tuning_results.iloc[0]["params"]
    best_train_oof_scores = next(scores for params, scores in tuning_oof_by_params if params == best_params)
    train_threshold_table = threshold_table(y_train, best_train_oof_scores).sort_values(metric_priority, ascending=False).reset_index(drop=True)
    tuned_threshold = float(train_threshold_table.iloc[0]["threshold"])

    fitted_model = fit_catboost_feature_set(features, best_params)
    held_out_scores = predict_scores(fitted_model, X_test_catboost[features].copy())
    held_out_metrics = binary_metrics(y_test, held_out_scores, threshold=tuned_threshold)

    row = {
        "model": model_label,
        "feature_count": len(features),
        "includes_sodium": "SODIUM" in features,
        "best_params": best_params,
        "threshold": tuned_threshold,
        **held_out_metrics,
    }
    matrix = pd.DataFrame(
        [[held_out_metrics["tn"], held_out_metrics["fp"]], [held_out_metrics["fn"], held_out_metrics["tp"]]],
        index=["Actual normal", "Actual abnormal"],
        columns=["Predicted normal", "Predicted abnormal"],
    )
    return row, matrix, train_threshold_table, tuning_results, fitted_model, held_out_scores

main_sodium_section_row, main_sodium_section_cm, main_sodium_threshold_table, main_sodium_tuning_results, main_sodium_section_model, main_sodium_scores = evaluate_catboost_sensitivity_feature_set(
    "Main CatBoost: BorutaSHAP accepted features only",
    main_accepted_features_for_sodium_section,
)
sodium_sensitivity_row, sodium_sensitivity_cm, sodium_sensitivity_threshold_table, sodium_sensitivity_tuning_results, sodium_sensitivity_eval_model, sodium_sensitivity_scores = evaluate_catboost_sensitivity_feature_set(
    "Sodium sensitivity CatBoost: accepted features + SODIUM",
    sodium_sensitivity_features,
)

sodium_sensitivity_comparison = pd.DataFrame([main_sodium_section_row, sodium_sensitivity_row])[
    [
        "model",
        "feature_count",
        "includes_sodium",
        "best_params",
        "threshold",
        "f1",
        "recall_sensitivity",
        "precision",
        "specificity",
        "balanced_accuracy",
        "accuracy",
        "roc_auc",
        "tn",
        "fp",
        "fn",
        "tp",
    ]
]

sodium_metric_delta = (
    sodium_sensitivity_comparison.set_index("model")
    .loc["Sodium sensitivity CatBoost: accepted features + SODIUM", ["f1", "recall_sensitivity", "precision", "specificity", "balanced_accuracy", "accuracy", "roc_auc"]]
    - sodium_sensitivity_comparison.set_index("model")
    .loc["Main CatBoost: BorutaSHAP accepted features only", ["f1", "recall_sensitivity", "precision", "specificity", "balanced_accuracy", "accuracy", "roc_auc"]]
).to_frame("sodium_minus_main")

sensitivity_tuning_results = pd.concat(
    [
        main_sodium_tuning_results.assign(model="Main CatBoost: BorutaSHAP accepted features only"),
        sodium_sensitivity_tuning_results.assign(model="Sodium sensitivity CatBoost: accepted features + SODIUM"),
    ],
    ignore_index=True,
)

print("Held-out sensitivity comparison. Hyperparameters and thresholds were selected using training OOF predictions only.")
display(sodium_sensitivity_comparison.round(4))
print("Main model confusion matrix")
display(main_sodium_section_cm)
print("Sodium sensitivity model confusion matrix")
display(sodium_sensitivity_cm)
print("Sensitivity minus main metric delta")
display(sodium_metric_delta.round(4))
print("Sensitivity tuning results")
display(sensitivity_tuning_results)

sodium_sensitivity_comparison.to_csv(SODIUM_OUTPUT_DIR / "sodium_sensitivity_model_comparison.csv", index=False)
sodium_metric_delta.to_csv(SODIUM_OUTPUT_DIR / "sodium_sensitivity_metric_delta.csv")
sensitivity_tuning_results.to_csv(SODIUM_OUTPUT_DIR / "sodium_sensitivity_tuning_results.csv", index=False)
main_sodium_section_cm.to_csv(SODIUM_OUTPUT_DIR / "main_model_confusion_matrix_for_sodium_comparison.csv")
sodium_sensitivity_cm.to_csv(SODIUM_OUTPUT_DIR / "sodium_sensitivity_confusion_matrix.csv")

### 6.5 Conclusion

The following conclusion is generated from the sodium sensitivity comparison and SHAP/dependence outputs above. Sodium remains outside the main accepted-feature CatBoost model unless future validation supports promoting it.

In [ ]:
from IPython.display import Markdown

comparison_indexed = sodium_sensitivity_comparison.set_index("model")
main_row = comparison_indexed.loc["Main CatBoost: BorutaSHAP accepted features only"]
sodium_row = comparison_indexed.loc["Sodium sensitivity CatBoost: accepted features + SODIUM"]

f1_delta = float(sodium_row["f1"] - main_row["f1"])
balanced_accuracy_delta = float(sodium_row["balanced_accuracy"] - main_row["balanced_accuracy"])
roc_auc_delta = float(sodium_row["roc_auc"] - main_row["roc_auc"])

sodium_display_values = sodium_shap_display_df["SODIUM"].astype(float).to_numpy()
valid_sodium_signal = np.isfinite(sodium_display_values) & np.isfinite(sodium_shap_values)
if valid_sodium_signal.sum() >= 3:
    sodium_shap_correlation = float(np.corrcoef(sodium_display_values[valid_sodium_signal], sodium_shap_values[valid_sodium_signal])[0, 1])
else:
    sodium_shap_correlation = np.nan

if abs(sodium_shap_correlation) >= 0.30:
    shap_interpretability_statement = (
        f"The sodium SHAP dependence shows a moderate relationship with model contribution "
        f"(correlation {sodium_shap_correlation:.2f}), which may be clinically interpretable with domain review."
    )
elif np.isfinite(sodium_shap_correlation):
    shap_interpretability_statement = (
        f"The sodium SHAP dependence appears weak or non-monotonic "
        f"(correlation {sodium_shap_correlation:.2f}), so clinical interpretability is limited in this run."
    )
else:
    shap_interpretability_statement = "The sodium SHAP dependence pattern could not be summarized reliably from the available held-out rows."

meaningful_gain = (f1_delta > 0.02) and (balanced_accuracy_delta > 0.02)
if meaningful_gain and abs(sodium_shap_correlation) >= 0.30:
    sodium_recommendation = "SODIUM may be discussed as an exploratory sensitivity-model feature, but it should still remain outside the main accepted-feature model pending validation."
elif meaningful_gain:
    sodium_recommendation = "SODIUM improves some held-out metrics in this run, but the SHAP pattern needs clinical review; keep it exploratory/tentative."
else:
    sodium_recommendation = "SODIUM should remain tentative and be discussed only as exploratory unless future validation shows consistent improvement."

sodium_conclusion_md = f"""
**Sodium sensitivity conclusion**

- Adding `SODIUM` changed held-out F1 by `{f1_delta:+.3f}`, balanced accuracy by `{balanced_accuracy_delta:+.3f}`, and ROC-AUC by `{roc_auc_delta:+.3f}` compared with the main BorutaSHAP accepted-feature CatBoost model.
- {shap_interpretability_statement}
- {sodium_recommendation}
- The main final CatBoost model is unchanged and continues to use BorutaSHAP accepted features only; this sodium analysis is exploratory and leakage-safe, with fitting and threshold tuning performed on training data only.
""".strip()

display(Markdown(sodium_conclusion_md))
Path(SODIUM_OUTPUT_DIR / "sodium_conclusion.md").write_text(sodium_conclusion_md, encoding="utf-8")
